# Expert Training with Knowledge Distillation (Model 2)

Trains per-corpus cross-encoders using supervised labels + KD from teacher pipeline.

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

## Step 2: Build Training Groups with RRF Fusion

Retrieves top-K candidates per query using weighted RRF over:
- **E5** (multilingual-e5-large): Dense semantic retrieval
- **TF-IDF (word)**: Unigram + bigram sparse retrieval
- **TF-IDF (char)**: Character 3-4gram sparse retrieval

Outputs train/val groups with ground-truth labels attached.

In [1]:
# ---------------------- Config ----------------------
import os, json, math, time, random, hashlib, gc
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer

CORPUS_JSONL   = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")
TRAIN_JSONL    = os.getenv("TRAIN_JSONL",  "/content/hsrc_train_augmented.jsonl")

# NEW: default to a fresh k80 output directory
OUTPUT_DIR     = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80")

E5_MODEL_NAME  = os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large")

# --- Retrieval sizes ---
K_E5           = int(os.getenv("K_E5", "80"))
RRF_POOL_MULT  = float(os.getenv("RRF_POOL_MULT", "3"))
RRF_POOL_MIN   = int(os.getenv("RRF_POOL_MIN", "150"))
RRF_K          = int(os.getenv("RRF_K", "60"))

# --- Fusion weights ---
WEIGHT_E5            = float(os.getenv("WEIGHT_E5", "0.6"))
WEIGHT_TFIDF         = float(os.getenv("WEIGHT_TFIDF", "0.4"))
WEIGHT_TFIDF_CHAR    = float(os.getenv("WEIGHT_TFIDF_CHAR", "0.15"))

# --- TF-IDF options ---
TFIDF_MAX_FEATS = int(os.getenv("TFIDF_MAX_FEATS", "300000"))
TFIDF_NGRAM_MIN = int(os.getenv("TFIDF_NGRAM_MIN", "1"))
TFIDF_NGRAM_MAX = int(os.getenv("TFIDF_NGRAM_MAX", "2"))

# --- Char TF-IDF ---
ENABLE_CHAR_TFIDF     = int(os.getenv("ENABLE_CHAR_TFIDF", "1"))
TFIDF_CHAR_MIN        = int(os.getenv("TFIDF_CHAR_MIN", "3"))
TFIDF_CHAR_MAX        = int(os.getenv("TFIDF_CHAR_MAX", "4"))
TFIDF_CHAR_MAX_FEATS  = int(os.getenv("TFIDF_CHAR_MAX_FEATS", "200000"))
TFIDF_CHAR_ANALYZER   = os.getenv("TFIDF_CHAR_ANALYZER", "char_wb")

VAL_SIZE       = float(os.getenv("VAL_SIZE", "0.6"))
SEED           = int(os.getenv("SEED", "42"))

SPLIT_VAL             = int(os.getenv("SPLIT_VAL", "1"))
FILTER_VAL_LEAKAGE    = int(os.getenv("FILTER_VAL_LEAKAGE", "0"))

# NEW: reuse previous split (train/val query uuid lists)
# If you set REUSE_SPLIT_DIR, we will look for train_query_uuids.txt and val_query_uuids.txt inside it.
REUSE_SPLIT_DIR = os.getenv("REUSE_SPLIT_DIR", "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/stage1")
TRAIN_QIDS_SRC  = os.getenv("TRAIN_QIDS_SRC", os.path.join(REUSE_SPLIT_DIR, "train_query_uuids.txt") if REUSE_SPLIT_DIR else "")
VAL_QIDS_SRC    = os.getenv("VAL_QIDS_SRC",   os.path.join(REUSE_SPLIT_DIR, "val_query_uuids.txt")   if REUSE_SPLIT_DIR else "")
STRICT_REUSE    = int(os.getenv("STRICT_REUSE", "1"))  # 1 = raise if any qids missing; 0 = warn and proceed

# Embedding cache
EMB_CACHE_DIR        = os.getenv("EMB_CACHE_DIR", "/content/e5_cache")
USE_MEMMAP_ON_LOAD   = bool(int(os.getenv("USE_MEMMAP_ON_LOAD", "1")))
Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Where to save stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(STAGE1_DIR).mkdir(parents=True, exist_ok=True)
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k{K_E5}.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k{K_E5}.jsonl"))
META_JSON          = os.getenv("STAGE1_META_JSON",   os.path.join(STAGE1_DIR, "meta.json"))

# NEW: paths for query UUID lists (saved for THIS run)
TRAIN_QIDS_TXT      = os.path.join(STAGE1_DIR, "train_query_uuids.txt")
VAL_QIDS_TXT        = os.path.join(STAGE1_DIR, "val_query_uuids.txt")
QID_SPLIT_JSON      = os.path.join(STAGE1_DIR, "query_uuid_splits.json")


# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] device={device}  K_FINAL={K_E5}  VAL_SIZE={VAL_SIZE}  SEED={SEED}  "
      f"SPLIT_VAL={SPLIT_VAL}  FILTER_VAL_LEAKAGE={FILTER_VAL_LEAKAGE}")
print(f"[PATHS] CORPUS={CORPUS_JSONL}  TRAIN={TRAIN_JSONL}  OUT={OUTPUT_DIR}  CACHE={EMB_CACHE_DIR}  STAGE1={STAGE1_DIR}")
print(f"[FUSION] RRF_POOL_MULT={RRF_POOL_MULT}  RRF_POOL_MIN={RRF_POOL_MIN}  RRF_K={RRF_K}  "
      f"WEIGHTS: E5={WEIGHT_E5} TFIDF(word)={WEIGHT_TFIDF} TFIDF(char)={WEIGHT_TFIDF_CHAR}")
print(f"[TFIDF(word)] max_features={TFIDF_MAX_FEATS} ngram=({TFIDF_NGRAM_MIN},{TFIDF_NGRAM_MAX})")
print(f"[TFIDF(char)] enabled={bool(ENABLE_CHAR_TFIDF)} analyzer={TFIDF_CHAR_ANALYZER} "
      f"ngram=({TFIDF_CHAR_MIN},{TFIDF_CHAR_MAX}) max_features={TFIDF_CHAR_MAX_FEATS}")

# ---------------------- IO ----------------------
def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid: corpus[uid] = o.get("passage") or o.get("text") or ""
    return corpus

def _as_int(x):
    if x is None: return 0
    s=str(x);  return int(s) if s.isdigit() else int(float(s))

def load_train(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            q   = o.get("query","")
            qid = o.get("query_uuid") or None
            case = o.get("case_name")  # keep this
            paras = o.get("paragraphs",{}) or {}
            labels= o.get("target_actions",{}) or {}
            gt = {}
            for i in range(1000):
                pk, lk = f"paragraph_{i}", f"target_action_{i}"
                if pk not in paras or lk not in labels: break
                puid = paras[pk].get("uuid") or paras[pk].get("id")
                rel  = _as_int(labels[lk])
                if puid: gt[puid] = rel
            rows.append({"query_uuid": qid, "query": q, "gt": gt, "case_name": case})
    return rows

corpus = load_corpus(CORPUS_JSONL)
print(f"[DATA] corpus docs={len(corpus):,}")
train_rows = load_train(TRAIN_JSONL)
print(f"[DATA] queries total={len(train_rows):,}")

# ---------------------- E5 retriever ----------------------
class E5Retriever:
    def __init__(self, name):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None),
                attn_implementation="sdpa"
            ).to(device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None)
            ).to(device)
        self.model.eval()

    @torch.no_grad()
    def embed(self, texts: List[str], is_query=False, bs=128) -> np.ndarray:
        pref = "query: " if is_query else "passage: "
        out = []
        for i in range(0, len(texts), bs):
            enc = self.tok([pref+t for t in texts[i:i+bs]], padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(device)
            h = self.model(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1)
            emb = (h*m).sum(1) / m.sum(1).clamp(min=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            out.append(emb.cpu())
        return torch.cat(out,0).numpy()

# ---------------------- Embedding cache helpers ----------------------
def _file_sig(p: Path):
    try:
        st = p.stat()
        return f"{p.name}|{st.st_size}|{int(st.st_mtime)}"
    except Exception:
        return f"{p.name}|0|0"

def _emb_cache_key(corpus_path: str, model_name: str, max_len: int, num_docs_hint: int = 0) -> str:
    p = Path(corpus_path)
    base = f"{_file_sig(p)}|{model_name}|L{max_len}|N{num_docs_hint}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:16]

def _emb_cache_paths(key: str):
    base = f"e5_{key}"
    e = os.path.join(EMB_CACHE_DIR, base + "_embeddings.npy")
    i = os.path.join(EMB_CACHE_DIR, base + "_ids.json")
    m = os.path.join(EMB_CACHE_DIR, base + "_meta.json")
    fx = os.path.join(EMB_CACHE_DIR, base + "_index.faiss")
    return e, i, m, fx

def load_embeddings_cache(corpus_path: str, model_name: str, max_len: int, expect_n: int):
    key = _emb_cache_key(corpus_path, model_name, max_len, expect_n)
    e, i, m, fx = _emb_cache_paths(key)
    if not (os.path.exists(e) and os.path.exists(i) and os.path.exists(m)):
        return None
    try:
        meta = json.load(open(m, "r", encoding="utf-8"))
        if meta.get("model_name") != model_name: return None
        ids = json.load(open(i, "r", encoding="utf-8"))
        embs = np.load(e, mmap_mode=("r" if USE_MEMMAP_ON_LOAD else None))
        if embs.shape[0] != len(ids): return None
        print(f"[CACHE] E5 embeddings restored from cache: {e} (shape={embs.shape}, memmap={USE_MEMMAP_ON_LOAD})")
        return {"embeddings": embs, "ids": ids, "meta": meta, "faiss_path": fx, "key": key}
    except Exception as ex:
        print("[CACHE] Failed to load cache:", ex)
        return None

def save_embeddings_cache(corpus_path: str, model_name: str, max_len: int, ids: List[str], embs: np.ndarray):
    key = _emb_cache_key(corpus_path, model_name, max_len, len(ids))
    e, i, m, fx = _emb_cache_paths(key)
    arr = np.asarray(embs, dtype=np.float32, order="C")
    np.save(e, arr)
    json.dump(list(ids), open(i, "w", encoding="utf-8"), ensure_ascii=False)
    meta = {"model_name": model_name, "num_documents": len(ids), "dim": int(arr.shape[1]),
            "corpus_path": str(corpus_path), "max_len": int(512)}
    json.dump(meta, open(m, "w", encoding="utf-8"))
    print(f"[CACHE] Saved embeddings: {e}  (ids: {i}, meta: {m})")
    return {"key": key, "emb_path": e, "ids_path": i, "meta_path": m, "faiss_path": fx}

# ---------------------- FAISS helpers (CPU-only) ----------------------
try:
    import faiss
    FAISS=True
except Exception as e:
    print("[FAISS] unavailable:", e)
    FAISS=False

def build_index(xb: np.ndarray):
    xb = np.asarray(xb, dtype=np.float32, order="C")
    idx = faiss.IndexFlatIP(xb.shape[1])   # CPU index
    idx.add(xb)
    print("[FAISS] Using CPU IndexFlatIP")
    return idx

def save_faiss_cpu_index(index, path: str):
    try:
        faiss.write_index(index, path)  # already CPU
        print(f"[CACHE] Saved FAISS CPU index → {path}")
    except Exception as e:
        print("[CACHE] Could not save FAISS index:", e)

def load_faiss_cpu(path: str):
    try:
        idx = faiss.read_index(path)
        print("[FAISS] Loaded CPU index.")
        return idx
    except Exception as e:
        print("[CACHE] Failed to load FAISS index:", e)
        return None

# ---------------------- Build / Load E5 corpus embeddings & index ----------------------
e5 = E5Retriever(E5_MODEL_NAME)
doc_ids = list(corpus.keys())
doc_texts_map = corpus  # uid -> text

# Try cache
cache = load_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, expect_n=len(doc_ids))
if cache is not None:
    cached_ids = cache["ids"]
    if len(cached_ids) == len(doc_ids):
        doc_ids = cached_ids
        doc_texts = [doc_texts_map[i] for i in doc_ids]
        X = cache["embeddings"]
        fx_path = cache["faiss_path"]
        if FAISS and os.path.exists(fx_path):
            index = load_faiss_cpu(fx_path)
        elif FAISS:
            index = build_index(X)
            save_faiss_cpu_index(index, fx_path)
        else:
            index = None
    else:
        print("[CACHE] Cached ids count mismatch; recomputing embeddings.")
        cache = None

if cache is None:
    doc_ids = list(corpus.keys())
    doc_texts = [corpus[d] for d in doc_ids]
    print("[E5] Embedding corpus …")
    X = e5.embed(doc_texts, is_query=False, bs=128)
    print(f"[E5] Corpus embeddings: {X.shape}")
    saved_meta = save_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, doc_ids, X)
    if FAISS:
        index = build_index(X)
        save_faiss_cpu_index(index, saved_meta["faiss_path"])
    else:
        index = None

# Fallback vector-matmul for search if FAISS missing
if not FAISS and isinstance(X, np.memmap):
    Xdot = np.array(X).T
elif not FAISS:
    Xdot = X.T

# ---------------------- TF-IDF (word) build ----------------------
print("[TFIDF] Fitting word TF-IDF on corpus …")
tfidf_vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATS,
                            ngram_range=(TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX))
tfidf_mat = tfidf_vec.fit_transform(doc_texts).astype(np.float32)  # CSR float32
print(f"[TFIDF] Matrix shape={tfidf_mat.shape} nnz={tfidf_mat.nnz:,}")

# ---------------------- TF-IDF (char) build ----------------------
if ENABLE_CHAR_TFIDF:
    print("[TFIDF-CHAR] Fitting char TF-IDF on corpus …")
    tfidf_char_vec = TfidfVectorizer(
        analyzer=TFIDF_CHAR_ANALYZER,
        ngram_range=(TFIDF_CHAR_MIN, TFIDF_CHAR_MAX),
        max_features=TFIDF_CHAR_MAX_FEATS,
        dtype=np.float32,
    )
    tfidf_char_mat = tfidf_char_vec.fit_transform(doc_texts).tocsr().astype(np.float32, copy=False)
    print(f"[TFIDF-CHAR] Matrix shape={tfidf_char_mat.shape} nnz={tfidf_char_mat.nnz:,}")
else:
    tfidf_char_vec, tfidf_char_mat = None, None
    print("[TFIDF-CHAR] Disabled.")

# ---------------------- RRF utilities ----------------------
def rrf_fuse(e5: List[Tuple[int, float]],
             lex: List[Tuple[int, float]],
             k: int,
             lex_c: List[Tuple[int, float]] = None) -> List[int]:
    """
    Fuse up to three ranked lists (indices in doc_ids) using weighted RRF.
    Accepts: e5 (pairs), lex word (pairs), optional lex char (pairs).
    """
    # Prepare rank maps
    sources = []
    weights = []
    if e5:
        sources.append({gi: r for r, (gi, _) in enumerate(e5)})
        weights.append(WEIGHT_E5)
    if lex:
        sources.append({gi: r for r, (gi, _) in enumerate(lex)})
        weights.append(WEIGHT_TFIDF)
    if lex_c:
        sources.append({gi: r for r, (gi, _) in enumerate(lex_c)})
        weights.append(WEIGHT_TFIDF_CHAR)

    if not sources:
        return []

    w_sum = sum(weights) if sum(weights) > 0 else 1.0
    norm_w = [w / w_sum for w in weights]

    universe = set().union(*[set(s.keys()) for s in sources])
    fused = []
    BIG = 10**9
    for gi in universe:
        s = 0.0
        for m, w in zip(sources, norm_w):
            r = m.get(gi, BIG)
            if r < BIG:
                s += float(w) / (RRF_K + r)
        fused.append((gi, s))

    fused.sort(key=lambda x: x[1], reverse=True)
    return [gi for gi, _ in fused[:k]]

# ---------------------- Per-source top-k (with scores) ----------------------
@torch.no_grad()
def e5_topk_with_scores(query: str, k: int):
    qv = e5.embed([query], is_query=True, bs=1).astype(np.float32)
    if FAISS:
        D, I = index.search(qv, min(k, len(doc_ids)))
        pairs = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i >= 0]
        return pairs  # already sorted by score desc
    else:
        sims = (qv @ Xdot)[0]
        ords = np.argsort(sims)[::-1][:k]
        return [(int(i), float(sims[i])) for i in ords]

def tfidf_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    qv = tfidf_vec.transform([query]).astype(np.float32)   # 1 x F (CSR float32)
    prod = (tfidf_mat @ qv.T)                              # N x 1 sparse
    scores = (prod.toarray().ravel()
              if hasattr(prod, "toarray") else np.asarray(prod).ravel())
    scores = scores.astype(np.float32, copy=False)
    take = min(k, scores.shape[0])
    if take <= 0:
        return []
    part = np.argpartition(scores, -take)[-take:]
    ords = part[np.argsort(scores[part])[::-1]]
    return [(int(i), float(scores[i])) for i in ords]

def tfidf_char_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    """Sparse top-k on char TF-IDF (no dense materialization)."""
    if not ENABLE_CHAR_TFIDF or tfidf_char_vec is None or tfidf_char_mat is None or k <= 0:
        return []
    v = tfidf_char_vec.transform([query])      # 1 x F CSR
    s = (tfidf_char_mat @ v.T).tocoo()         # N x 1 sparse
    if s.nnz == 0:
        return []
    take = min(k, s.nnz)
    part = np.argpartition(s.data, -take)[-take:]
    ords = part[np.argsort(s.data[part])[::-1]]
    return [(int(s.row[i]), float(s.data[i])) for i in ords]

def fused_topk_ids(query: str, k_final: int) -> List[str]:
    pool_k = max(int(k_final * RRF_POOL_MULT), RRF_POOL_MIN)
    e5_list        = e5_topk_with_scores(query, pool_k)
    tfidf_list     = tfidf_topk_with_scores(query, pool_k)
    tfidf_char_list= tfidf_char_topk_with_scores(query, pool_k)
    fused_idx  = rrf_fuse(e5_list, tfidf_list, k_final, lex_c=tfidf_char_list)
    return [doc_ids[i] for i in fused_idx]
# ---------------------- Build top-K (fused) groups for every query ----------------------
# Each group: {'query', 'query_uuid', 'case_name', 'pids', 'texts', 'labels'}
groups = []
retrieved_relevant = 0
total_relevant = 0
queries_with_relevant = 0
for row in train_rows:
    pids   = fused_topk_ids(row["query"], K_E5)
    texts  = [doc_texts_map[pid] for pid in pids]
    labels = [int(row["gt"].get(pid, 0)) for pid in pids]  # unlabeled -> 0

    groups.append({
        "query": row["query"],
        "query_uuid": row.get("query_uuid"),
        "case_name": row.get("case_name"),
        "pids": pids,
        "texts": texts,
        "labels": labels
    })

    rel_total = sum(1 for v in row["gt"].values() if v > 0)
    if rel_total:
        queries_with_relevant += 1
        total_relevant += rel_total
        retrieved_relevant += sum(1 for pid in pids if row["gt"].get(pid, 0) > 0)

print(f"[GROUPS] built fused (E5 ⊕ TF-IDF[word]{' ⊕ TF-IDF[char]' if ENABLE_CHAR_TFIDF else ''}) top-{K_E5} for all queries.")
if total_relevant:
    recall_at_k = retrieved_relevant / total_relevant
    print(f"[METRICS] Recall@{K_E5} = {recall_at_k:.4f} ({retrieved_relevant}/{total_relevant}) "
          f"over {queries_with_relevant} queries with relevance.")
else:
    print(f"[METRICS] Recall@{K_E5} undefined (no relevant labels).")

# ---------------------- Upper-bound NDCG@P (perfect reranker) — per corpus ----------------------
from collections import defaultdict

NDCG_P = int(os.getenv("NDCG_P", "20"))

def _dcg_at_p(rels, p):
    s = 0.0
    for i, r in enumerate(rels[:p], start=1):
        s += (2**int(r) - 1) / math.log2(i + 1)
    return s

per_sum = defaultdict(float)
per_cnt = defaultdict(int)
overall_sum, overall_cnt = 0.0, 0

# Use train_rows (full GT) for IDCG; groups (retrieved pool) for DCG upper bound
for row, g in zip(train_rows, groups):
    # Ideal DCG from all known labels for this query
    gt_rels = sorted([int(v) for v in row["gt"].values()], reverse=True)
    idcg = _dcg_at_p(gt_rels, NDCG_P)
    if idcg <= 0:
        continue  # skip queries with no positives

    # Perfect reranker within the retrieved pool (your fused top-K_E5)
    pool_rels = sorted([int(l) for l in g["labels"]], reverse=True)
    ub_ndcg = _dcg_at_p(pool_rels, NDCG_P) / idcg

    tag = g.get("case_name") or "UNKNOWN"
    per_sum[tag] += ub_ndcg
    per_cnt[tag] += 1
    overall_sum += ub_ndcg
    overall_cnt += 1

print(f"[UB][NDCG] Perfect-reranker upper bound NDCG@{NDCG_P} by corpus:")
for tag in sorted(per_sum.keys()):
    mean = per_sum[tag] / per_cnt[tag] if per_cnt[tag] else float("nan")
    print(f"    {tag:12s}: {mean:.4f}  (n={per_cnt[tag]})")
if overall_cnt:
    print(f"[UB][NDCG] Overall: {overall_sum/overall_cnt:.4f}  (n={overall_cnt})")

# ---------------------- Free heavy objects to reduce RAM/VRAM ----------------------
try: del e5
except Exception: pass
try: del index
except Exception: pass
for _name in ("X", "Xdot", "doc_texts", "doc_texts_map", "tfidf_mat", "tfidf_vec",
              "tfidf_char_mat", "tfidf_char_vec"):
    if _name in globals(): globals()[_name] = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[CLEANUP] Freed E5/FAISS & TF-IDF; cleared CUDA cache.")

# ---------------------- Split by query + optional leakage filter (REUSE if provided) ----------------------
def _read_lines(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]

def _filter_train_docs(train_groups, banned):
    kept, drop_items, drop_groups = [], 0, 0
    for g in train_groups:
        mask = [pid not in banned for pid in g["pids"]]
        if not any(mask):
            drop_groups += 1
            continue
        pids  = [p for p,m in zip(g["pids"], mask) if m]
        texts = [t for t,m in zip(g["texts"], mask) if m]
        labs  = [l for l,m in zip(g["labels"],mask) if m]
        drop_items += (len(g["pids"]) - len(pids))
        kept.append({
            "query": g["query"],
            "query_uuid": g.get("query_uuid"),
            "case_name": g.get("case_name"),
            "pids": pids, "texts": texts, "labels": labs
        })
    return kept, drop_items, drop_groups

reuse_available = (TRAIN_QIDS_SRC and os.path.exists(TRAIN_QIDS_SRC)) and \
                  (VAL_QIDS_SRC   and os.path.exists(VAL_QIDS_SRC))

if reuse_available:
    print(f"[SPLIT] Reusing split from:\n"
          f"       train_qids: {TRAIN_QIDS_SRC}\n"
          f"       val_qids:   {VAL_QIDS_SRC}")

    src_train_qids = set(_read_lines(TRAIN_QIDS_SRC))
    src_val_qids   = set(_read_lines(VAL_QIDS_SRC))

    all_qids = {g.get("query_uuid") for g in groups if g.get("query_uuid") is not None}
    missing_train = src_train_qids - all_qids
    missing_val   = src_val_qids   - all_qids

    if missing_train:
        msg = f"[SPLIT][WARN] {len(missing_train)} train qids not present in current groups (first 5): {list(sorted(missing_train))[:5]}"
        print(msg)
        if STRICT_REUSE: raise RuntimeError(msg)
    if missing_val:
        msg = f"[SPLIT][WARN] {len(missing_val)} val qids not present in current groups (first 5): {list(sorted(missing_val))[:5]}"
        print(msg)
        if STRICT_REUSE: raise RuntimeError(msg)

    groups_train = [g for g in groups if g.get("query_uuid") in src_train_qids]
    groups_val   = [g for g in groups if g.get("query_uuid") in src_val_qids]
    print(f"[SPLIT] train groups={len(groups_train)}  val groups={len(groups_val)}  (reused split)")

    if FILTER_VAL_LEAKAGE:
        val_doc_set = set(pid for gv in groups_val for pid in gv["pids"])
        groups_train, n_drop_items, n_drop_groups = _filter_train_docs(groups_train, val_doc_set)
        print(f"[LEAKAGE] removed {n_drop_items} train items; dropped {n_drop_groups} fully-overlapping train groups")
    else:
        print("[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).")

else:
    # Original random split (no reuse files found)
    if SPLIT_VAL:
        idxs = list(range(len(groups)))
        random.shuffle(idxs)
        cut = int(len(groups)*(1.0-VAL_SIZE))
        groups_train = [groups[i] for i in idxs[:cut]]
        groups_val   = [groups[i] for i in idxs[cut:]]
        print(f"[SPLIT] train groups={len(groups_train)}  val groups={len(groups_val)}  (VAL_SIZE={VAL_SIZE})")

        if FILTER_VAL_LEAKAGE:
            val_doc_set = set(pid for gv in groups_val for pid in gv["pids"])
            groups_train, n_drop_items, n_drop_groups = _filter_train_docs(groups_train, val_doc_set)
            print(f"[LEAKAGE] removed {n_drop_items} train items; dropped {n_drop_groups} fully-overlapping train groups")
        else:
            print("[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).")
    else:
        groups_train = groups
        groups_val   = []
        print(f"[SPLIT] no validation split (SPLIT_VAL=0): using ALL {len(groups_train)} groups for training.")

# ---------------------- Save stage-1 outputs ----------------------
def _write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def _write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for ln in lines:
            f.write(str(ln) + "\n")

_write_jsonl(GROUPS_TRAIN_JSONL, groups_train)
_write_jsonl(GROUPS_VAL_JSONL, groups_val)

# --- save the query UUIDs of train/val splits ---
train_qids = [g.get("query_uuid") for g in groups_train if g.get("query_uuid") is not None]
val_qids   = [g.get("query_uuid") for g in groups_val   if g.get("query_uuid") is not None]

_write_lines(TRAIN_QIDS_TXT, train_qids)
_write_lines(VAL_QIDS_TXT, val_qids)

with open(QID_SPLIT_JSON, "w", encoding="utf-8") as f:
    json.dump({"train_query_uuids": train_qids, "val_query_uuids": val_qids}, f, ensure_ascii=False, indent=2)

meta = {
    # include both keys so downstream code can read either
    "K_FINAL": K_E5,
    "K_E5": K_E5,
    "VAL_SIZE": VAL_SIZE, "SEED": SEED,
    "split_val": int(SPLIT_VAL), "filter_val_leakage": int(FILTER_VAL_LEAKAGE),
    "corpus_path": CORPUS_JSONL, "train_path": TRAIN_JSONL,
    "e5_model": E5_MODEL_NAME, "groups_train": len(groups_train), "groups_val": len(groups_val),
    "fusion": {
        "type": "RRF",
        "RRF_K": RRF_K,
        "RRF_POOL_MULT": RRF_POOL_MULT,
        "RRF_POOL_MIN": RRF_POOL_MIN,
        "weights": {
            "e5": WEIGHT_E5,
            "tfidf_word": WEIGHT_TFIDF,
            "tfidf_char": WEIGHT_TFIDF_CHAR
        }
    },
    "tfidf": {
        "word": {"max_features": TFIDF_MAX_FEATS, "ngram": [TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX]},
        "char": {
            "enabled": int(ENABLE_CHAR_TFIDF),
            "analyzer": TFIDF_CHAR_ANALYZER,
            "ngram": [TFIDF_CHAR_MIN, TFIDF_CHAR_MAX],
            "max_features": TFIDF_CHAR_MAX_FEATS
        }
    },
    # NEW: include counts for convenience
    "train_query_uuids_path": TRAIN_QIDS_TXT,
    "val_query_uuids_path": VAL_QIDS_TXT,
    "query_uuid_splits_json": QID_SPLIT_JSON,
    "train_query_uuids_count": len(train_qids),
    "val_query_uuids_count": len(val_qids),
}
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"[SAVE] groups_train → {GROUPS_TRAIN_JSONL}")
print(f"[SAVE] groups_val   → {GROUPS_VAL_JSONL}")
print(f"[SAVE] train_query_uuids.txt → {TRAIN_QIDS_TXT}  (n={len(train_qids)})")
print(f"[SAVE] val_query_uuids.txt   → {VAL_QIDS_TXT}    (n={len(val_qids)})")
print(f"[SAVE] query_uuid_splits.json→ {QID_SPLIT_JSON}")
print(f"[SAVE] meta         → {META_JSON}")


[CONFIG] device=cuda  K_FINAL=80  VAL_SIZE=0.6  SEED=42  SPLIT_VAL=1  FILTER_VAL_LEAKAGE=0
[PATHS] CORPUS=/content/hsrc_corpus.jsonl  TRAIN=/content/hsrc_train_augmented.jsonl  OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80  CACHE=/content/e5_cache  STAGE1=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80/stage1
[FUSION] RRF_POOL_MULT=3.0  RRF_POOL_MIN=150  RRF_K=60  WEIGHTS: E5=0.6 TFIDF(word)=0.4 TFIDF(char)=0.15
[TFIDF(word)] max_features=300000 ngram=(1,2)
[TFIDF(char)] enabled=True analyzer=char_wb ngram=(3,4) max_features=200000
[DATA] corpus docs=127,731
[DATA] queries total=2,034


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


[E5] Embedding corpus …
[E5] Corpus embeddings: (127731, 1024)
[CACHE] Saved embeddings: /content/e5_cache/e5_9a4542dd1df10b48_embeddings.npy  (ids: /content/e5_cache/e5_9a4542dd1df10b48_ids.json, meta: /content/e5_cache/e5_9a4542dd1df10b48_meta.json)
[FAISS] Using CPU IndexFlatIP
[CACHE] Saved FAISS CPU index → /content/e5_cache/e5_9a4542dd1df10b48_index.faiss
[TFIDF] Fitting word TF-IDF on corpus …
[TFIDF] Matrix shape=(127731, 300000) nnz=21,476,500
[TFIDF-CHAR] Fitting char TF-IDF on corpus …
[TFIDF-CHAR] Matrix shape=(127731, 200000) nnz=106,512,380
[GROUPS] built fused (E5 ⊕ TF-IDF[word] ⊕ TF-IDF[char]) top-80 for all queries.
[METRICS] Recall@80 = 0.8502 (12998/15289) over 2021 queries with relevance.
[UB][NDCG] Perfect-reranker upper bound NDCG@20 by corpus:
    mafat_retrieval_knesset_corpus: 0.8242  (n=475)
    mafat_retrieval_kz_corpus: 0.9124  (n=804)
    mafat_retrieval_wikipedia_corpus: 0.9782  (n=742)
[UB][NDCG] Overall: 0.9158  (n=2021)
[CLEANUP] Freed E5/FAISS & TF-IDF

## Step 3: Add Pseudo-Labels and Handle Near-Duplicates

Augments training groups with:
1. **Pseudo-labels** from LLM relabeling (`crit_relabel_pipeline.ipynb`) - adds labels for previously zero-positive queries
2. **Near-duplicate handling** - removes duplicate passages that could cause label conflicts

Outputs enriched `groups_train_k80_pseudo_dedup.jsonl` for training.

In [5]:
import os, json
from pathlib import Path
from typing import Dict, List, DefaultDict
from collections import defaultdict

# ---------- Paths ----------
STAGE1_DIR = os.getenv("STAGE1_DIR", "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80/stage1")
CORPUS_JSONL = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")

GROUPS_TRAIN_IN = os.getenv(
    "GROUPS_TRAIN_JSONL_IN",
    os.path.join(STAGE1_DIR, "groups_train_k80.jsonl")
)

PSEUDO_JSONL = os.getenv(
    "PSEUDO_JSONL",
    "/content/llm_labels_combined_ge4_ge3.jsonl"    # your file with query_uuid, doc_id, label, ...
)

DUPS_JSONL = os.getenv(
    "DUPS_JSONL",
    "/content/label4_near_dups.jsonl"  # file with label4_pid/dup_pid pairs as you showed
)

GROUPS_TRAIN_OUT = os.getenv(
    "GROUPS_TRAIN_JSONL_OUT",
    os.path.join(STAGE1_DIR, "groups_train_k80_pseudo_dedup.jsonl")
)

print(f"[PATHS] STAGE1_DIR           = {STAGE1_DIR}")
print(f"[PATHS] CORPUS_JSONL         = {CORPUS_JSONL}")
print(f"[PATHS] GROUPS_TRAIN_IN      = {GROUPS_TRAIN_IN}")
print(f"[PATHS] PSEUDO_JSONL         = {PSEUDO_JSONL}")
print(f"[PATHS] DUPS_JSONL           = {DUPS_JSONL}")
print(f"[PATHS] GROUPS_TRAIN_OUT     = {GROUPS_TRAIN_OUT}")


# ---------- IO helpers ----------
def read_jsonl(path: str) -> List[dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            if ln.strip():
                items.append(json.loads(ln))
    return items

def write_jsonl(path: str, items: List[dict]):
    with open(path, "w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            if not ln.strip():
                continue
            o = json.loads(ln)
            uid = o.get("uuid") or o.get("id")
            if uid:
                txt = o.get("passage") or o.get("text") or ""
                corpus[uid] = txt
    return corpus


# ---------- Load inputs ----------
print("[LOAD] corpus…")
corpus = load_corpus(CORPUS_JSONL)
print(f"[LOAD] corpus docs={len(corpus):,}")

print("[LOAD] training groups…")
groups_train = read_jsonl(GROUPS_TRAIN_IN)
print(f"[LOAD] groups_train={len(groups_train):,}")

print("[LOAD] pseudo labels…")
pseudo_records = read_jsonl(PSEUDO_JSONL)
print(f"[LOAD] pseudo records={len(pseudo_records):,}")

print("[LOAD] duplicate pairs…")
dups_records = read_jsonl(DUPS_JSONL)
print(f"[LOAD] duplicate records={len(dups_records):,}")


# ---------- Build maps ----------
# Map: query_uuid -> list of pseudo records
pseudo_by_qid: DefaultDict[str, List[dict]] = defaultdict(list)
for rec in pseudo_records:
    qid = rec.get("query_uuid")
    if not qid:
        continue
    pseudo_by_qid[qid].append(rec)

print(f"[MAP] queries with pseudo labels={len(pseudo_by_qid):,}")

# Map: (case_name, query) -> set of duplicate pids (dup_pid)
# We only care about dup_pid here; label4_pid is the "canonical" positive.
dup_pids_by_case_query: DefaultDict[tuple, set] = defaultdict(set)
for rec in dups_records:
    case_name = rec.get("case_name") or ""
    query = rec.get("query") or ""
    dup_pid = rec.get("dup_pid")
    if not dup_pid or not query:
        continue
    key = (case_name, query)
    dup_pids_by_case_query[key].add(dup_pid)

print(f"[MAP] (case_name,query) keys with dup pids={len(dup_pids_by_case_query):,}")


# ---------- Attach pseudo labels, skipping wiki duplicates ----------
aug_groups: List[dict] = []
num_added = 0
num_upgraded = 0
num_skipped_dup = 0
num_skipped_missing_text = 0

for g in groups_train:
    qid = g.get("query_uuid")
    case_name = g.get("case_name") or ""
    query_text = g.get("query") or ""
    key = (case_name, query_text)

    is_wiki = ("wikipedia" in case_name.lower()) or ("mafat_retrieval_wikipedia_corpus" in case_name.lower())
    dup_pids_for_q = dup_pids_by_case_query.get(key, set())

    pids = list(g["pids"])
    texts = list(g["texts"])
    labels = list(g["labels"])

    # quick index for docs already in group
    pid2idx = {pid: idx for idx, pid in enumerate(pids)}

    if qid and qid in pseudo_by_qid:
        for prec in pseudo_by_qid[qid]:
            doc_id = prec.get("doc_id") or prec.get("paragraph_uuid")
            if not doc_id:
                continue

            # pick label (top-level), fallback to gpt5/gemini if you want
            lab_raw = prec.get("label")
            if lab_raw is None:
                lab_raw = (prec.get("gpt5") or {}).get("label", None)
            if lab_raw is None:
                lab_raw = (prec.get("gemini") or {}).get("label", None)
            if lab_raw is None:
                continue

            try:
                lab = int(lab_raw)
            except Exception:
                try:
                    lab = int(float(lab_raw))
                except Exception:
                    continue

            if lab <= 0:
                continue

            # For wikipedia: skip pseudo label-4 that is marked as a duplicate
            if is_wiki and lab == 4 and doc_id in dup_pids_for_q:
                num_skipped_dup += 1
                continue

            # If doc already in group, just upgrade label if higher
            if doc_id in pid2idx:
                idx = pid2idx[doc_id]
                old_lab = int(labels[idx])
                if lab > old_lab:
                    labels[idx] = int(lab)
                    num_upgraded += 1
                continue

            # Need passage text to attach a new item
            txt = corpus.get(doc_id)
            if not txt:
                num_skipped_missing_text += 1
                continue

            # Attach as new pseudo-labeled doc
            pid2idx[doc_id] = len(pids)
            pids.append(doc_id)
            texts.append(txt)
            labels.append(int(lab))
            num_added += 1

    # update group
    g["pids"] = pids
    g["texts"] = texts
    g["labels"] = labels
    aug_groups.append(g)

print("\n[STATS] Pseudo-label attachment with dup filtering:")
print(f"  Added new docs:                      {num_added}")
print(f"  Upgraded labels on existing docs:    {num_upgraded}")
print(f"  Skipped duplicates via DUPS_JSONL:   {num_skipped_dup}")
print(f"  Skipped (missing passage in corpus): {num_skipped_missing_text}")

# ---------- Save augmented groups ----------
write_jsonl(GROUPS_TRAIN_OUT, aug_groups)
print(f"[SAVE] Augmented train groups → {GROUPS_TRAIN_OUT}")


[PATHS] STAGE1_DIR           = /content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80/stage1
[PATHS] CORPUS_JSONL         = /content/hsrc_corpus.jsonl
[PATHS] GROUPS_TRAIN_IN      = /content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80/stage1/groups_train_k80.jsonl
[PATHS] PSEUDO_JSONL         = /content/llm_labels_combined_ge4_ge3.jsonl
[PATHS] DUPS_JSONL           = /content/label4_near_dups.jsonl
[PATHS] GROUPS_TRAIN_OUT     = /content/ce_ft/bge-reranker-v2-m3_t50_fp16_k80/stage1/groups_train_k80_pseudo_dedup.jsonl
[LOAD] corpus…
[LOAD] corpus docs=127,731
[LOAD] training groups…
[LOAD] groups_train=1,728
[LOAD] pseudo labels…
[LOAD] pseudo records=647
[LOAD] duplicate pairs…
[LOAD] duplicate records=72
[MAP] queries with pseudo labels=307
[MAP] (case_name,query) keys with dup pids=52

[STATS] Pseudo-label attachment with dup filtering:
  Added new docs:                      34
  Upgraded labels on existing docs:    328
  Skipped duplicates via DUPS_JSONL:   14
  Skipped (missing passage in corpus

## Step 4: Mixed-Label Passages Analysis (Diagnostics)

Analyzes passages that have **different labels for different queries** within the same corpus.

**Purpose**: Identify potential label noise or ambiguous passages that may hurt training.

Outputs CSV files with mixed-label passages and example queries per corpus.

In [4]:
# === Mixed-label passages per corpus (hsrc_train.jsonl) ===
# Finds passages (UUIDs) that are positive (>0) for some queries and negative (0) for others
# within the SAME corpus (case_name). Streams the JSONL to keep memory low.

import os, json, gc
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Iterable, Optional
import pandas as pd

# ---------- CONFIG ----------
HSRC_TRAIN_JSONL = os.getenv("HSRC_TRAIN_JSONL", "/content/hsrc_train_augmented.jsonl")  # <-- set your path here
OUTPUT_DIR       = os.getenv("OUTPUT_DIR", "/content/hsrc_overlap_checks")
BATCH_SIZE       = int(os.getenv("BATCH_SIZE", "1024"))
NUM_EXAMPLES     = int(os.getenv("NUM_EXAMPLES", "3"))   # how many pos/neg example queries per mixed passage
WRITE_EXAMPLES   = True                                  # toggle second pass to collect example queries

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ---------- HELPERS ----------
def _iter_jsonl_batches(path: str, batch_size: int):
    buf = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            buf.append(json.loads(line))
            if len(buf) >= batch_size:
                yield buf
                buf = []
    if buf:
        yield buf

def _case_to_slug(case_name: str) -> str:
    low = (case_name or "").lower()
    if "knesset" in low: return "knesset"
    if "wikipedia" in low or "wiki" in low: return "wiki"
    if "kz" in low or "kol" in low or "zchut" in low: return "kz"
    return (case_name or "unknown")

def _labels_from_target_actions(ta_obj: dict) -> List[int]:
    # ta_obj: {"target_action_0": "0", ...} or ints; return list[int] length 20 in index order
    pairs = []
    for k, v in ta_obj.items():
        try:
            idx = int(k.split("_")[-1])
        except Exception:
            continue
        try:
            lab = int(v)
        except Exception:
            lab = int(float(v))
        pairs.append((idx, lab))
    pairs.sort(key=lambda x: x[0])
    return [lab for _, lab in pairs]

def _pids_from_paragraphs(pars_obj: dict) -> List[str]:
    # pars_obj: {"paragraph_0":{"uuid":...,"passage":...}, ...}
    pairs = []
    for k, v in pars_obj.items():
        try:
            idx = int(k.split("_")[-1])
        except Exception:
            continue
        pid = (v or {}).get("uuid")
        pairs.append((idx, pid))
    pairs.sort(key=lambda x: x[0])
    return [pid for _, pid in pairs]

# ---------- PASS 1: count pos/neg per (corpus, passage UUID) ----------
print(f"[PASS1] Scanning {HSRC_TRAIN_JSONL} ...")
pos_counts: Dict[str, Counter] = defaultdict(Counter)  # slug -> Counter(pid -> #pos)
neg_counts: Dict[str, Counter] = defaultdict(Counter)  # slug -> Counter(pid -> #neg)
n_queries = 0
n_rows = 0

for batch in _iter_jsonl_batches(HSRC_TRAIN_JSONL, BATCH_SIZE):
    for obj in batch:
        n_rows += 1
        slug = _case_to_slug(obj.get("case_name", "unknown"))
        labs = _labels_from_target_actions(obj.get("target_actions", {}))
        pids = _pids_from_paragraphs(obj.get("paragraphs", {}))
        # defensive: lengths may mismatch in edge cases
        L = min(len(labs), len(pids))
        if L == 0:
            continue
        n_queries += 1
        for i in range(L):
            pid = pids[i]
            if not pid:
                continue
            lab = int(labs[i])
            if lab > 0:
                pos_counts[slug][pid] += 1
            else:
                neg_counts[slug][pid] += 1
    gc.collect()

print(f"[PASS1] Done. queries={n_queries:,}  rows={n_rows:,}")

# ---------- Summaries + CSV of mixed passages ----------
def _summarize_and_save(slug: str):
    pos = pos_counts.get(slug, Counter())
    neg = neg_counts.get(slug, Counter())
    all_pids = set(pos.keys()) | set(neg.keys())
    mixed = [pid for pid in all_pids if pos[pid] > 0 and neg[pid] > 0]
    pos_only = [pid for pid in all_pids if pos[pid] > 0 and neg[pid] == 0]
    neg_only = [pid for pid in all_pids if pos[pid] == 0 and neg[pid] > 0]

    print(f"\n[SUMMARY:{slug}] unique_passages={len(all_pids):,}  "
          f"mixed={len(mixed):,} ({(100*len(mixed)/max(1,len(all_pids))):.2f}%)  "
          f"pos_only={len(pos_only):,}  neg_only={len(neg_only):,}")

    rows = []
    for pid in mixed:
        pc, nc = pos[pid], neg[pid]
        rows.append({"slug": slug, "pid": pid, "pos_cnt": pc, "neg_cnt": nc,
                     "total": pc + nc, "pos_frac": pc / float(pc + nc)})
    df = pd.DataFrame(rows).sort_values(["total", "pos_cnt"], ascending=[False, False])
    out_csv = Path(OUTPUT_DIR) / f"mixed_passages__{slug}.csv"
    df.to_csv(out_csv, index=False)
    print(f"[WRITE] mixed_passages__{slug}.csv  rows={len(df)}  → {out_csv}")
    return set(mixed)

mixed_sets_by_slug: Dict[str, set] = {}
for slug in sorted(set(list(pos_counts.keys()) + list(neg_counts.keys()))):
    mixed_sets_by_slug[slug] = _summarize_and_save(slug)

# ---------- PASS 2 (optional): collect example queries for mixed passages ----------
if WRITE_EXAMPLES:
    print("\n[PASS2] Collecting up to "
          f"{NUM_EXAMPLES} pos and {NUM_EXAMPLES} neg example queries per mixed passage...")
    # For speed, precompute which PIDs we need per slug
    need_pid = {slug: set(mixed_sets_by_slug[slug]) for slug in mixed_sets_by_slug}
    # Map: slug -> pid -> {"pos":[(qid, query)], "neg":[(qid, query)]}
    examples = {slug: defaultdict(lambda: {"pos": [], "neg": []}) for slug in mixed_sets_by_slug}

    # Helper to decide if we still need examples for that pid and polarity
    def _need_more(slug, pid, pol):
        return len(examples[slug][pid][pol]) < NUM_EXAMPLES

    for batch in _iter_jsonl_batches(HSRC_TRAIN_JSONL, BATCH_SIZE):
        for obj in batch:
            slug = _case_to_slug(obj.get("case_name", "unknown"))
            if slug not in need_pid or not need_pid[slug]:
                continue
            labs = _labels_from_target_actions(obj.get("target_actions", {}))
            pids = _pids_from_paragraphs(obj.get("paragraphs", {}))
            qtext = obj.get("query", "")
            qid   = obj.get("query_uuid", "")
            L = min(len(labs), len(pids))
            for i in range(L):
                pid = pids[i]
                if not pid or pid not in need_pid[slug]:
                    continue
                pol = "pos" if int(labs[i]) > 0 else "neg"
                if _need_more(slug, pid, pol):
                    examples[slug][pid][pol].append((qid, qtext))
            # Stop early if finished for this batch of slugs
            for s in list(need_pid.keys()):
                # If all pids for slug have enough examples, clear to speed up
                done_all = True
                for pid in list(need_pid[s]):
                    if _need_more(s, pid, "pos") or _need_more(s, pid, "neg"):
                        done_all = False
                        break
                if done_all:
                    need_pid[s].clear()
        gc.collect()

    # Write examples per corpus
    for slug in examples:
        rows = []
        for pid in mixed_sets_by_slug.get(slug, []):
            pos_ex = examples[slug][pid]["pos"][:NUM_EXAMPLES]
            neg_ex = examples[slug][pid]["neg"][:NUM_EXAMPLES]
            rows.append({
                "slug": slug,
                "pid": pid,
                "pos_examples": " || ".join([f"{qid}: {q[:200].replace('\n',' ')}" for qid, q in pos_ex]) if pos_ex else "",
                "neg_examples": " || ".join([f"{qid}: {q[:200].replace('\n',' ')}" for qid, q in neg_ex]) if neg_ex else "",
            })
        df = pd.DataFrame(rows)
        out_csv = Path(OUTPUT_DIR) / f"mixed_passages_examples__{slug}.csv"
        df.to_csv(out_csv, index=False)
        print(f"[WRITE] mixed_passages_examples__{slug}.csv  rows={len(df)}  → {out_csv}")

print("\n[DONE] Feasibility check complete.")
print("Interpretation: If 'mixed' is non-trivial per corpus and examples look reasonable,")
print("then you can construct training groups that show the SAME passage as relevant for some queries")
print("and irrelevant for others (strong supervision for a reranker).")


[PASS1] Scanning /content/hsrc_train_augmented.jsonl ...
[PASS1] Done. queries=2,034  rows=2,034

[SUMMARY:knesset] unique_passages=6,463  mixed=1,049 (16.23%)  pos_only=1,995  neg_only=3,419
[WRITE] mixed_passages__knesset.csv  rows=1049  → /content/hsrc_overlap_checks/mixed_passages__knesset.csv

[SUMMARY:kz] unique_passages=7,067  mixed=2,081 (29.45%)  pos_only=2,053  neg_only=2,933
[WRITE] mixed_passages__kz.csv  rows=2081  → /content/hsrc_overlap_checks/mixed_passages__kz.csv

[SUMMARY:wiki] unique_passages=12,158  mixed=903 (7.43%)  pos_only=2,917  neg_only=8,338
[WRITE] mixed_passages__wiki.csv  rows=903  → /content/hsrc_overlap_checks/mixed_passages__wiki.csv

[PASS2] Collecting up to 3 pos and 3 neg example queries per mixed passage...
[WRITE] mixed_passages_examples__knesset.csv  rows=1049  → /content/hsrc_overlap_checks/mixed_passages_examples__knesset.csv
[WRITE] mixed_passages_examples__kz.csv  rows=2081  → /content/hsrc_overlap_checks/mixed_passages_examples__kz.csv
[WRIT

## Step 5: Load Teacher Scores

Copy pre-computed teacher scores from Google Drive for Knowledge Distillation.

Teacher scores are generated by the KD pipeline (`kd_pipeline_final.ipynb`) using 4-way RRF retrieval + Voyage reranking + 3-way reranker fusion.

In [28]:
cp -r /content/drive/MyDrive/mafat_hsrc/teacher_scores_new .


## Step 6: Train Per-Corpus Cross-Encoders with Knowledge Distillation

Fine-tunes **BGE-reranker-v2-m3** per corpus using combined loss:
- **Supervised loss** (α=0.7): ListNet on ground-truth labels with per-corpus r_tables
- **KD loss** (β=0.3): ListNet on teacher scores from KD pipeline (Voyage + reranker RRF fusion)

**Per-corpus config**:
| Corpus | Base Model | Epochs | LR | Max Length |
|--------|------------|--------|------|------------|
| Wiki | BGE-reranker-v2-m3 | 2 | 8e-6 | 320 |
| KZ | KZ-pretrained CE | 3 | 8e-6 | 448 |
| Knesset | BGE-reranker-v2-m3 | 4 | 8e-6 | 496 |

Saves best checkpoint per corpus (by validation NDCG@20).

In [ ]:
# === Cell 2: Stage 2 — fine-tune corpus-specific cross-encoders with per-corpus params + KD ===
import os, json, math, time, random, gc
from pathlib import Path
from typing import List, Dict, Tuple, Union
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup

# ---------------------- Base Config (defaults, can be overridden per corpus) ----------------------
OUTPUT_DIR      = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og")
CE_MODEL_NAME   = os.getenv("CE_MODEL_NAME","BAAI/bge-reranker-v2-m3")   # default CE for all

# (NEW) Optional path to your CSV-pretrained KZ CE (will auto-fallback to OUTPUT_DIR/ce_kz_pretrain/best)
KZ_PRETRAIN_CE_DIR = os.getenv("KZ_PRETRAIN_CE_DIR", "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/ce_kz_pretrain/best").strip()

# Training/eval defaults (per-corpus overrides below)
EPOCHS          = int(os.getenv("EPOCHS", "2"))
LR              = float(os.getenv("LR", "1.5e-5"))
BATCH_GROUPS    = int(os.getenv("BATCH_GROUPS", "2"))
GRAD_ACCUM      = int(os.getenv("GRAD_ACCUM", "1"))
MAX_LEN         = int(os.getenv("MAX_LEN", "384"))
WEIGHT_DECAY    = float(os.getenv("WEIGHT_DECAY", "0.02"))
CLIP_NORM       = float(os.getenv("CLIP_NORM", "1.0"))
TAU             = float(os.getenv("TAU", "1.1"))
VAL_EVAL_K      = int(os.getenv("VAL_EVAL_K", "20"))
LOSS_LABELED_ONLY  = int(os.getenv("LOSS_LABELED_ONLY", "0"))  # 1: train only on labeled (>0), 0: listwise on all

# KD settings (teacher = Voyage / RRF scores)
ALPHA_SUP       = float(os.getenv("ALPHA_SUP", "0.7"))   # supervised ListNet weight
BETA_KD         = float(os.getenv("BETA_KD",   "0.3"))   # KD ListNet weight
T_KD            = float(os.getenv("T_KD",      "1.8"))   # teacher temperature
KD_MISSING_SENTINEL = float(os.getenv("KD_MISSING_SENTINEL", "nan"))

# Eval/pretokenization
EVAL_BATCH_PAIRS   = int(os.getenv("EVAL_BATCH_PAIRS", "128"))
PAD_TO_MULTIPLE_OF = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))

SEED = int(os.getenv("SEED", "42"))

# Stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
# These may be updated automatically from stage1 meta below
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k70.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k70.jsonl"))

# Teacher files (produced by RRF+Voy generator)
TEACHER_DIR   = os.getenv("TEACHER_DIR", os.path.join("/content/", "teacher_scores_new"))
TEACHER_FILEPAT = os.getenv("TEACHER_FILEPAT", "{corpus}_teacher_rrf_voy25_k100.jsonl")  # adjust pattern if needed

# ---------------------- Which corpora to run ----------------------
# "all" or any subset list, e.g., ["kz"], ["kz","knesset"], ["wiki"]
RUN_CORPORA = os.getenv("RUN_CORPORA", "knesset")   # <-- change here in notebook if you want a subset

# ---------------------- Per-corpus overrides ----------------------
# Put any field here to override defaults for that corpus.
# Supported keys include: CE_MODEL_NAME, EPOCHS, LR, BATCH_GROUPS, GRAD_ACCUM, MAX_LEN,
# WEIGHT_DECAY, CLIP_NORM, TAU, VAL_EVAL_K, LOSS_LABELED_ONLY, EVAL_BATCH_PAIRS, PAD_TO_MULTIPLE_OF,
# ALPHA_SUP, BETA_KD, T_KD
PER_CORPUS_OVERRIDES = {
  # Wikipedia: short passages, many label-4s, already strong baseline → keep it efficient & stable
  "wiki": {
    "EPOCHS": 2,
    "LR": 8e-6,
    "MAX_LEN": 320,          # passages: mean≈134, p95≈238 → 320 comfortably covers
    "BATCH_GROUPS": 2,
    "GRAD_ACCUM": 1,
    "TAU": 0.9,
    "EVAL_BATCH_PAIRS": 192, # faster eval on the A10
    "ALPHA_SUP": 0.65,
    "BETA_KD": 0.5,
    "T_KD": 1.8,
  },

  # KZ (Kol-Zchut): longest passages + heavy positives tail → give more context & a touch more temp
  "kz":  {
    "EPOCHS": 3,
    "LR": 8e-6,
    "MAX_LEN": 448,
    "BATCH_GROUPS": 1,
    "GRAD_ACCUM": 2,
    "TAU": 1.0,
    "EVAL_BATCH_PAIRS": 64,
    "ALPHA_SUP": 0.7,
    "BETA_KD": 0.2,
    "T_KD": 1.8,
    # "CE_MODEL_NAME":  (we set this below once we resolve your pretrain path)
  },

  # Knesset: hardest domain, medium length, fewer label-4s → steadier LR, one extra epoch, slightly higher τ
  "knesset":{
    "EPOCHS": 4,
    "LR": 8e-6,
    "MAX_LEN": 488,
    "BATCH_GROUPS": 1,
    "GRAD_ACCUM": 2,
    "TAU": 0.95,
    "EVAL_BATCH_PAIRS": 32,
    "ALPHA_SUP": 0.8,
    "BETA_KD": 0.1,
    "T_KD": 2.2,
  },
}

# ---------------------- Corpus keys & aliases ----------------------
CORPUS_KEYS = {
    "mafat_retrieval_wikipedia_corpus": "wiki",
    "mafat_retrieval_kz_corpus":        "kz",
    "mafat_retrieval_knesset_corpus":   "knesset",
}
SLUG_TO_CORPUSKEY = {v: k for k, v in CORPUS_KEYS.items()}
# Accept a common misspelling
ALIASES = {"kenesset": "knesset"}

# For teacher file naming
SLUG_TO_TEACHER_CORPUSNAME = {
    "wiki": "wikipedia",
    "kz": "kz",
    "knesset": "knesset",
}

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Allow TF32 for speed on Ampere
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"[CONFIG] device={device}")
print(f"[PATHS] OUT={OUTPUT_DIR}  STAGE1={STAGE1_DIR}  TEACHER_DIR={TEACHER_DIR}")

# ---------------------- Per-corpus ratings (r_table) ----------------------
# Defaults based on your no-training search (label order: 0..4)
R_TABLES_DEFAULT = {
    "wiki":    [0.0, 1.12116936, 5.2045328, 15.2433052, 37.70000983],
    "kz":      [0.0, 1.28305769, 6.25935032, 21.0831492, 42.58293821],
    "knesset": [0.0, 1.12116936, 4.75205225, 16.28861888, 40.28529796],
}

def _load_r_tables(json_path: str) -> Dict[str, List[float]]:
    """
    Load per-corpus r_tables from a JSON file if it exists; otherwise fall back to defaults.
    Expected JSON structure:
    {
      "wiki":    { "r_table": [0.0, ...] },
      "kz":      { "r_table": [0.0, ...] },
      "knesset": { "r_table": [0.0, ...] },
    }
    """
    rt = {k: v[:] for k, v in R_TABLES_DEFAULT.items()}
    try:
        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            for slug in ("wiki", "kz", "knesset"):
                if slug in data and "r_table" in data[slug]:
                    arr = [float(x) for x in data[slug]["r_table"]]
                    # sanity: 5 entries, label 0 gain ~= 0
                    if len(arr) == 5 and abs(arr[0]) < 1e-12:
                        rt[slug] = arr
            print(f"[R_TABLES] loaded from {json_path}")
        else:
            print(f"[R_TABLES] file not found, using defaults")
    except Exception as e:
        print(f"[R_TABLES] failed to load ({e}); using defaults")
    return rt

IDEAL_R_JSON = os.path.join(OUTPUT_DIR, "ideal_r_no_training.json")
R_TABLES = _load_r_tables(IDEAL_R_JSON)

# ---------------------- IO ----------------------
def _read_jsonl(path):
    items=[]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

# Auto-match Stage-1 K for group paths if meta exists (support both K_E5 and K_FINAL)
_meta_path = os.path.join(STAGE1_DIR, "meta.json")
_stage1_meta = None
if os.path.exists(_meta_path):
    try:
        _stage1_meta = json.load(open(_meta_path, "r", encoding="utf-8"))
        _k_meta = _stage1_meta.get("K_E5") or _stage1_meta.get("K_FINAL")
        if _k_meta:
            if "GROUPS_TRAIN_JSONL" not in os.environ:
                GROUPS_TRAIN_JSONL = os.path.join(STAGE1_DIR, f"groups_train_k{_k_meta}.jsonl")
            if "GROUPS_VAL_JSONL" not in os.environ:
                GROUPS_VAL_JSONL   = os.path.join(STAGE1_DIR, f"groups_val_k{_k_meta}.jsonl")
    except Exception:
        pass

groups_train = _read_jsonl(GROUPS_TRAIN_JSONL)
groups_val   = _read_jsonl(GROUPS_VAL_JSONL)
print(f"[DATA] train groups={len(groups_train)}  val groups={len(groups_val)}")

# Ensure groups have case_name; Stage-1 updated to include it, but fall back if needed
def _q2case_from_train_json(stage1_meta):
    mp = {}
    tp = stage1_meta.get("train_path") if stage1_meta else None
    if tp and os.path.exists(tp):
        with open(tp, "r", encoding="utf-8") as f:
            for ln in f:
                if not ln.strip(): continue
                o = json.loads(ln)
                q = o.get("query",""); cn = o.get("case_name") or "unknown"
                if q and q not in mp: mp[q] = cn
    return mp

_q2case = _q2case_from_train_json(_stage1_meta)

def _attach_case(groups):
    for g in groups:
        if "case_name" not in g or not g["case_name"]:
            g["case_name"] = _q2case.get(g.get("query",""), "unknown")
    return groups

groups_train = _attach_case(groups_train)
groups_val   = _attach_case(groups_val)

# Skip groups with zero positives (speeds training)
_before = len(groups_train)
groups_train = [g for g in groups_train if any(l > 0 for l in g["labels"])]
print(f"[DATA] filtered train groups with no positives: {_before - len(groups_train)} dropped; {len(groups_train)} remain")

# ---------------------- (NEW) Resolve CSV-pretrained KZ CE path ----------------------
def _has_model_dir(p: str) -> bool:
    if not p or not os.path.isdir(p): return False
    for f in ("config.json", "model.safetensors", "pytorch_model.bin"):
        if os.path.exists(os.path.join(p, f)):
            return True
    return False

_default_kz_pretrain = os.path.join(OUTPUT_DIR, "ce_kz_pretrain", "best")
_resolved_kz_start = None
if _has_model_dir(KZ_PRETRAIN_CE_DIR):
    _resolved_kz_start = KZ_PRETRAIN_CE_DIR
elif _has_model_dir(_default_kz_pretrain):
    _resolved_kz_start = _default_kz_pretrain

if _resolved_kz_start:
    PER_CORPUS_OVERRIDES.setdefault("kz", {})["CE_MODEL_NAME"] = _resolved_kz_start
    print(f"[INIT] KZ will start from CSV-pretrained CE: {_resolved_kz_start}")
else:
    print("[INIT] KZ will start from base CE_MODEL_NAME (no CSV-pretrain dir found).")

# ---------------------- Teacher loader & attach (KD) ----------------------
def _load_teacher_map_for_slug(slug: str) -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Returns: teacher[qid] -> {
        'score_by_doc': {doc_id: score},
        'rank_by_doc':  {doc_id: rank}
    }
    """
    corpusname = SLUG_TO_TEACHER_CORPUSNAME[slug]
    tpath = os.path.join(TEACHER_DIR, TEACHER_FILEPAT.format(corpus=corpusname))
    if not os.path.exists(tpath):
        print(f"[KD][WARN] teacher file not found for {slug}: {tpath}")
        return {}

    mp = {}
    n = 0
    for rec in _read_jsonl(tpath):
        qid   = rec.get("query_uuid")
        pids  = [str(p) for p in (rec.get("pids") or [])]
        scores = rec.get("teacher_scores") or []
        if not qid or not pids:
            continue

        score_by_doc = {}
        rank_by_doc  = {}
        for idx, pid in enumerate(pids):
            rank_by_doc[pid] = idx
            if idx < len(scores) and scores[idx] is not None:
                score_by_doc[pid] = float(scores[idx])

        mp[qid] = {
            "score_by_doc": score_by_doc,
            "rank_by_doc": rank_by_doc,
        }
        n += 1

    print(f"[KD] loaded teacher for {slug}: {n} queries from {tpath}")
    return mp


def _attach_teacher_scores(groups: List[dict], slug: str) -> None:
    """
    Adds g['teacher_scores'] (list[float or NaN]) aligned with g['pids'] for each group.
    """
    tmap = _load_teacher_map_for_slug(slug)
    touched, covered = 0, 0

    for g in groups:
        qid = g.get("query_uuid")
        if not qid:
            g["teacher_scores"] = [float("nan")] * len(g["pids"])
            continue

        rec  = tmap.get(qid, {})
        smap = rec.get("score_by_doc", {})
        ts   = []
        hit  = 0
        for pid in g["pids"]:
            s = smap.get(str(pid))
            if s is None:
                # missing → NaN by default (masked in KD loss)
                if math.isnan(KD_MISSING_SENTINEL):
                    ts.append(float("nan"))
                else:
                    ts.append(float(KD_MISSING_SENTINEL))
            else:
                ts.append(float(s))
                hit += 1

        g["teacher_scores"] = ts
        touched += 1
        covered += (hit > 0)

    print(f"[KD] attached teacher_scores to {touched} groups; "
          f"groups with ≥1 teacher score: {covered}")


def compute_teacher_val_metrics(groups: List[dict], slug: str, k: int) -> Dict[str, float]:
    """
    Compute teacher-only NDCG@k on validation:
      - 'scores': rank by Voyage score (missing → bottom)
      - 'ranks' : rank by teacher rank order (missing → bottom)
    Also returns mean doc coverage and groups_with_any_teacher count.
    """
    tmap = _load_teacher_map_for_slug(slug)
    nd_scores, nd_ranks = [], []
    doc_cov_fracs = []
    groups_with_any = 0

    for g in groups:
        qid  = g.get("query_uuid")
        pids = g["pids"]
        labs = g["labels"]
        rec  = tmap.get(qid, {})
        smap = rec.get("score_by_doc", {})
        rmap = rec.get("rank_by_doc", {})

        present = sum(1 for pid in pids if str(pid) in smap or str(pid) in rmap)
        if len(pids) > 0:
            doc_cov_fracs.append(present / len(pids))
        if present > 0:
            groups_with_any += 1

        gt = {pid: int(l) for pid, l in zip(pids, labs)}

        # by scores
        svals = [smap.get(str(pid), None) for pid in pids]
        svals_num = [(-1e30 if v is None else float(v)) for v in svals]
        order_scores = [pid for pid, _ in sorted(zip(pids, svals_num), key=lambda x: x[1], reverse=True)]
        nd_scores.append(ndcg_at_k(order_scores, gt, k))

        # by ranks
        rvals = [rmap.get(str(pid), 10**9) for pid in pids]
        order_ranks = [pid for pid, _ in sorted(zip(pids, rvals), key=lambda x: x[1])]
        nd_ranks.append(ndcg_at_k(order_ranks, gt, k))

    def _mean(x): return float(np.mean(x)) if len(x) else 0.0
    return {
        "ndcg_scores": _mean(nd_scores),
        "ndcg_ranks": _mean(nd_ranks),
        "doc_coverage_mean": _mean(doc_cov_fracs),
        "groups_with_any": int(groups_with_any),
        "groups_total": int(len(groups)),
    }

# ---------------------- Dataset / Collate ----------------------
class ListwiseDataset(torch.utils.data.Dataset):
    """
    Returns:
      query, texts, labels(human), pids, teacher_scores (list[float or NaN] or None)
    """
    def __init__(self, groups, use_teacher: bool = False):
        self.groups = groups
        self.use_teacher = use_teacher

    def __len__(self): return len(self.groups)

    def __getitem__(self, i):
        g = self.groups[i]
        t_scores = g.get("teacher_scores") if self.use_teacher else None
        return g["query"], g["texts"], g["labels"], g["pids"], t_scores


def collate_listwise(batch, tokenizer, max_len=512, r_table=None):
    """
    If r_table is provided (len=5, for labels 0..4), use it; otherwise fall back to 2^y-1 gains.
    Returns:
      enc, gains_tensor, spans, pids, teacher_scores_tensor
    """
    Q, P, gains, spans, PIDS, TSC = [], [], [], [], [], []
    cur = 0

    use_custom = isinstance(r_table, (list, tuple)) and len(r_table) == 5

    for q, texts, labs, pids, t_scores in batch:
        n = len(texts)
        Q += [q] * n
        P += texts

        if use_custom:
            gains.extend([float(r_table[int(l)]) for l in labs])
        else:
            gains.extend([float((2**int(l)) - 1) for l in labs])

        PIDS += pids
        spans.append((cur, cur + n))
        cur += n

        if t_scores is None:
            TSC += [float("nan")] * n
        else:
            TSC += [float(x) for x in t_scores]

    enc = tokenizer(
        Q, P,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

    gains_t = torch.tensor(gains, dtype=torch.float32)
    tsc_t   = torch.tensor(TSC,   dtype=torch.float32)

    return enc, gains_t, spans, PIDS, tsc_t

# ---------------------- Loss ----------------------
def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0:
            continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0


def listnet_loss_labeled_only(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        pos = g > 0
        if pos.sum() == 0:
            continue
        p_t = g[pos] / (g[pos].sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e][pos] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0


def listnet_kd_loss(scores: torch.Tensor,
                    t_scores: torch.Tensor,
                    spans,
                    T_kd: float = 1.8,
                    tau_student: float = 1.0):
    """
    KD ListNet (KL) using teacher per-item scores within each group.
    Ignores items whose t_scores are NaN/inf.
    """
    scores = scores.float()
    t_scores = t_scores.float()
    loss_terms = []

    for s, e in spans:
        ts = t_scores[s:e]
        mask = torch.isfinite(ts)
        if not torch.any(mask):
            continue

        ts = ts[mask]
        # stabilize
        ts = ts - ts.max()

        p_t = torch.softmax(ts / T_kd, dim=0)
        log_q = torch.log_softmax(scores[s:e][mask] / tau_student, dim=0)
        loss_terms.append(-(p_t * log_q).sum())

    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ---------------------- Eval helpers ----------------------
def _dcg_at_k(rels, k=20):
    return sum(((2**r-1)/math.log2(i+2) for i,r in enumerate(rels[:k])), 0.0)

def ndcg_at_k(ids_sorted: List[str], gt_map: Dict[str,int], k=20):
    rels = [gt_map.get(pid, 0) for pid in ids_sorted[:k]]
    dcg  = _dcg_at_k(rels, k)
    idcg = _dcg_at_k(sorted(gt_map.values(), reverse=True), k)
    return 0.0 if idcg==0.0 else float(dcg/idcg)

@torch.no_grad()
def evaluate_ndcg_groups(model, loader_or_pretok: Union[dict, torch.utils.data.DataLoader], k=20) -> List[float]:
    model.eval()
    # Fast pre-tokenized path
    if isinstance(loader_or_pretok, dict) and "enc" in loader_or_pretok:
        enc_full = loader_or_pretok["enc"]
        spans    = loader_or_pretok["spans"]
        rels_all = loader_or_pretok["rels"]

        if not spans or rels_all.numel() == 0:
            return []

        N = enc_full["input_ids"].size(0)
        scores = torch.empty(N, dtype=torch.float32)

        start = 0
        while start < N:
            end = min(start + params_runtime["EVAL_BATCH_PAIRS"], N)  # uses runtime params (set below)
            sl = slice(start, end)
            inputs = {k: v[sl].to(device, non_blocking=True) for k, v in enc_full.items()}

            ctx = (torch.autocast(device_type="cuda", dtype=torch.float16)
                   if device=="cuda" else torch.cpu.amp.autocast(enabled=False))
            with torch.inference_mode(), ctx:
                logits = model(**inputs).logits
                if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
                elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
                else: s = logits.view(-1).float()
            scores[sl] = s.detach().cpu()
            start = end

        denom = 1.0 / np.log2(np.arange(2, params_runtime["VAL_EVAL_K"] + 2))
        s_np = scores.numpy(); r_np = rels_all.numpy()
        ndcgs = []
        for (st, ed) in spans:
            group_scores = s_np[st:ed]
            group_rels   = r_np[st:ed]
            if group_rels.size == 0:
                ndcgs.append(0.0); continue
            order = np.argsort(-group_scores)
            rel_sorted = group_rels[order][:params_runtime["VAL_EVAL_K"]]
            dcg = ((np.power(2.0, rel_sorted, dtype=np.float64) - 1.0) * denom[:len(rel_sorted)]).sum()
            ideal = np.sort(group_rels)[::-1][:params_runtime["VAL_EVAL_K"]]
            idcg = ((np.power(2.0, ideal, dtype=np.float64) - 1.0) * denom[:len(ideal)]).sum()
            ndcgs.append(0.0 if idcg <= 0.0 else float(dcg / idcg))
        return ndcgs

    # Slow path (kept for completeness)
    ndcgs=[]
    for enc_cpu, gains_cpu, spans, pids in loader_or_pretok:
        enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        logits = model(**enc).logits
        if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
        elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
        else: s = logits.view(-1).float()
        scores = s.detach().cpu().numpy().tolist()
        for (st,ed) in spans:
            group_pids   = pids[st:ed]
            group_scores = scores[st:ed]
            g_local      = gains_cpu[st:ed].tolist()
            rels_local = []
            for g in g_local:
                if g <= 0: rels_local.append(0)
                elif math.isclose(g,1.0):  rels_local.append(1)
                elif math.isclose(g,3.0):  rels_local.append(2)
                elif math.isclose(g,7.0):  rels_local.append(3)
                elif math.isclose(g,15.0): rels_local.append(4)
                else: rels_local.append(int(round(math.log2(g+1))))
            gt = {pid: rel for pid, rel in zip(group_pids, rels_local)}
            ranked = [pid for pid,_ in sorted(zip(group_pids, group_scores), key=lambda x:x[1], reverse=True)]
            ndcgs.append(ndcg_at_k(ranked, gt, k=params_runtime["VAL_EVAL_K"]))
    return ndcgs

def _mean_or_zero(vals): return float(np.mean(vals)) if len(vals) > 0 else 0.0

def build_val_pretok(groups, tokenizer, max_len=512, pad_multi=8):
    all_input_ids, all_attn, all_ttids = [], [], []
    all_rels, all_pids, spans = [], [], []
    cur = 0
    for g in groups:
        q, texts, labels, pids = g["query"], g["texts"], g["labels"], g["pids"]
        enc = tokenizer(
            [q] * len(texts), texts,
            padding="max_length", truncation=True, max_length=max_len,
            pad_to_multiple_of=pad_multi, return_tensors="pt",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
        if "token_type_ids" in enc: all_ttids.append(enc["token_type_ids"])
        all_rels.extend([int(l) for l in labels]); all_pids.extend(pids)
        spans.append((cur, cur + len(texts))); cur += len(texts)

    if all_input_ids:
        input_ids = torch.cat(all_input_ids, dim=0)
        attention_mask = torch.cat(all_attn, dim=0)
        enc_full = {"input_ids": input_ids, "attention_mask": attention_mask}
        if len(all_ttids) > 0: enc_full["token_type_ids"] = torch.cat(all_ttids, dim=0)
        for k in list(enc_full.keys()):
            enc_full[k] = enc_full[k].pin_memory()
        rels = torch.tensor(all_rels, dtype=torch.int16)
    else:
        L = max_len
        enc_full = {"input_ids": torch.empty(0, L, dtype=torch.long),
                    "attention_mask": torch.empty(0, L, dtype=torch.long)}
        rels = torch.empty(0, dtype=torch.int16)
    return {"enc": enc_full, "spans": spans, "pids": all_pids, "rels": rels}

# ---------------------- Helpers: save size ----------------------
def _dir_size_bytes(path: str) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try: total += os.path.getsize(fp)
            except OSError: pass
    return total

def _human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    i = 0; v = float(n)
    while v >= 1024.0 and i < len(units)-1: v /= 1024.0; i += 1
    return f"{v:.2f} {units[i]}"

def _save_fp16_and_report(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    orig_dtype = next(model.parameters()).dtype
    try:
        model.to(dtype=torch.float16)
        model.save_pretrained(out_dir, safe_serialization=True)  # safetensors
        tokenizer.save_pretrained(out_dir)
    finally:
        model.to(dtype=orig_dtype)
    sz = _dir_size_bytes(out_dir)
    print(f"[SAVE] Checkpoint saved (fp16) → {out_dir}  |  folder size = {_human_bytes(sz)}")

# ---------------------- Filters & param plumbing ----------------------
def _filter_by_corpus(groups, corpus_key: str):
    return [g for g in groups if (g.get("case_name") == corpus_key)]

def _resolve_run_corpora(run_spec):
    if isinstance(run_spec, str):
        run_spec = run_spec.strip().lower()
        if run_spec == "all":
            return list(SLUG_TO_CORPUSKEY.keys())
        run_spec = ALIASES.get(run_spec, run_spec)
        return [run_spec] if run_spec in SLUG_TO_CORPUSKEY else []
    elif isinstance(run_spec, (list, tuple, set)):
        slugs = []
        for x in run_spec:
            y = ALIASES.get(str(x).lower(), str(x).lower())
            if y in SLUG_TO_CORPUSKEY: slugs.append(y)
        return list(dict.fromkeys(slugs))  # de-dup, preserve order
    else:
        return []

BASE_PARAMS = {
    "CE_MODEL_NAME": CE_MODEL_NAME,
    "EPOCHS": EPOCHS,
    "LR": LR,
    "BATCH_GROUPS": BATCH_GROUPS,
    "GRAD_ACCUM": GRAD_ACCUM,
    "MAX_LEN": MAX_LEN,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "CLIP_NORM": CLIP_NORM,
    "TAU": TAU,
    "VAL_EVAL_K": VAL_EVAL_K,
    "LOSS_LABELED_ONLY": int(LOSS_LABELED_ONLY),
    "EVAL_BATCH_PAIRS": EVAL_BATCH_PAIRS,
    "PAD_TO_MULTIPLE_OF": PAD_TO_MULTIPLE_OF,
    "ALPHA_SUP": ALPHA_SUP,
    "BETA_KD": BETA_KD,
    "T_KD": T_KD,
}

def _params_for_slug(slug: str) -> Dict[str, Union[int, float, str]]:
    p = BASE_PARAMS.copy()
    if slug in PER_CORPUS_OVERRIDES:
        for k, v in PER_CORPUS_OVERRIDES[slug].items():
            p[k] = v
    return p

# ---------------------- Train ONE corpus-specific CE ----------------------
def train_one_corpus(slug: str) -> Dict[str, float]:
    corpus_key = SLUG_TO_CORPUSKEY[slug]
    out_dir = os.path.join(OUTPUT_DIR, f"ce_{slug}")
    os.makedirs(out_dir, exist_ok=True)

    # Hydrate per-corpus params
    global params_runtime
    params_runtime = _params_for_slug(slug)  # used inside eval fn as well
    print(f"\n[SETUP:{slug}] params={params_runtime}")

    # Choose r_table for this corpus (fallback to defaults, then to 2^y-1)
    r_table_for_slug = R_TABLES.get(slug)
    if r_table_for_slug is None:
        print(f"[R_TABLES:{slug}] missing; falling back to defaults")
        r_table_for_slug = R_TABLES_DEFAULT.get(slug, [0.0, 1.0, 3.0, 7.0, 15.0])
    print(f"[R_TABLES:{slug}] {r_table_for_slug}")

    # Split groups by corpus
    tr = _filter_by_corpus(groups_train, corpus_key)
    va = _filter_by_corpus(groups_val,   corpus_key)
    print(f"[DATA:{slug}] train={len(tr)}  val={len(va)}  out={out_dir}")
    if len(tr) == 0 or len(va) == 0:
        print(f"[WARN:{slug}] Skipping (no data).")
        return {"baseline": 0.0, "best": 0.0}

    # Attach teacher scores for KD on train groups
    _attach_teacher_scores(tr, slug)

    # Optional: teacher-only VAL baselines
    try:
        t_val = compute_teacher_val_metrics(va, slug, params_runtime["VAL_EVAL_K"])
        print(f"[VAL:{slug}][TEACHER] NDCG@{params_runtime['VAL_EVAL_K']} "
              f"scores={t_val['ndcg_scores']:.4f} "
              f"| ranks={t_val['ndcg_ranks']:.4f} "
              f"| doc_cov={100.0*t_val['doc_coverage_mean']:.2f}% "
              f"| groups_any={t_val['groups_with_any']}/{t_val['groups_total']}")
    except Exception as e:
        print(f"[VAL:{slug}][TEACHER] metrics skipped ({e})")

    tok = AutoTokenizer.from_pretrained(params_runtime["CE_MODEL_NAME"])
    # Fresh model per corpus (starts from pretrain path if provided)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            params_runtime["CE_MODEL_NAME"], trust_remote_code=True, attn_implementation="sdpa"
        ).to(device)
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(
            params_runtime["CE_MODEL_NAME"], trust_remote_code=True
        ).to(device)
    model = model.float()

    # Loaders
    train_ds = ListwiseDataset(tr, use_teacher=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=params_runtime["BATCH_GROUPS"], shuffle=True,
        collate_fn=lambda b: collate_listwise(b, tok, params_runtime["MAX_LEN"], r_table_for_slug),
        pin_memory=torch.cuda.is_available(), num_workers=2 if torch.cuda.is_available() else 0,
        persistent_workers=torch.cuda.is_available()
    )

    pretok_val = build_val_pretok(va, tok, max_len=params_runtime["MAX_LEN"],
                                  pad_multi=params_runtime["PAD_TO_MULTIPLE_OF"])

    # Optim / sched
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params_runtime["LR"],
        weight_decay=params_runtime["WEIGHT_DECAY"], eps=1e-8, betas=(0.9, 0.999)
    )
    updates_per_epoch = max(1, math.ceil(len(train_loader) / max(1, params_runtime["GRAD_ACCUM"])))
    num_update_steps  = max(1, updates_per_epoch * params_runtime["EPOCHS"])
    warmup_ratio = 0.10
    warmup = max(1, int(warmup_ratio * num_update_steps))
    lr_end_frac = 0.10
    lr_end = max(params_runtime["LR"] * lr_end_frac, 1e-8)
    scheduler = get_polynomial_decay_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup,
        num_training_steps=num_update_steps,
        lr_end=lr_end,
        power=1.0
    )

    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # Baseline
    print(f"\n[VAL:{slug}] evaluating pretrained CE on VAL …")
    baseline_list = evaluate_ndcg_groups(model, pretok_val, k=params_runtime["VAL_EVAL_K"])
    baseline = _mean_or_zero(baseline_list)
    print(f"[VAL:{slug}] Baseline NDCG@{params_runtime['VAL_EVAL_K']}: {baseline:.4f}")

    best = baseline
    compute_sup_loss = (listnet_loss_labeled_only if int(params_runtime["LOSS_LABELED_ONLY"])
                        else listnet_loss)

    grad_accum = max(1, int(params_runtime["GRAD_ACCUM"]))

    for ep in range(1, params_runtime["EPOCHS"]+1):
        model.train()
        running, micro = 0.0, 0
        t0 = time.perf_counter()

        for step, (enc_cpu, gains_cpu, spans, _pids, t_scores_cpu) in enumerate(train_loader, 1):
            enc      = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
            gains    = gains_cpu.to(device, non_blocking=True)
            t_scores = t_scores_cpu.to(device, non_blocking=True)

            if use_amp:
                ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
            else:
                class _NoOp:
                    def __enter__(self): pass
                    def __exit__(self, *args): return False
                ctx = _NoOp()

            with ctx:
                logits = model(**enc).logits
                if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
                elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
                else: s = logits.view(-1).float()

                # Supervised labeled-only ListNet (doesn't directly punish unlabeled docs)
                L_sup = compute_sup_loss(s, gains, spans, tau=params_runtime["TAU"])

                # KD ListNet from teacher
                L_kd = listnet_kd_loss(
                    s, t_scores, spans,
                    T_kd=params_runtime.get("T_KD", T_KD),
                    tau_student=params_runtime["TAU"],
                )

                loss = (params_runtime.get("ALPHA_SUP", ALPHA_SUP) * L_sup +
                        params_runtime.get("BETA_KD",  BETA_KD)  * L_kd)

                # scale loss for gradient accumulation
                loss = loss / grad_accum

            micro += 1

            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Do optimizer step when we've accumulated enough steps, or at the very end
            do_step = (step % grad_accum == 0) or (step == len(train_loader))

            if do_step:
                if use_amp:
                    if params_runtime["CLIP_NORM"] > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), params_runtime["CLIP_NORM"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    if params_runtime["CLIP_NORM"] > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), params_runtime["CLIP_NORM"])
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            running += float(loss.item())  # note: this is per-microbatch loss / grad_accum
            if step % 50 == 0 or step == len(train_loader):
                elapsed = time.perf_counter() - t0
                print(f"[train:{slug} e{ep}/{params_runtime['EPOCHS']}] step {step}/{len(train_loader)} "
                      f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                      f"steps/s={step/max(1e-6,elapsed):.2f}")

        # Validate for this corpus
        val_list = evaluate_ndcg_groups(model, pretok_val, k=params_runtime["VAL_EVAL_K"])
        val_ndcg = _mean_or_zero(val_list)
        print(f"[VAL:{slug}] epoch {ep}  NDCG@{params_runtime['VAL_EVAL_K']}={val_ndcg:.4f}  (baseline {baseline:.4f})")

        if val_ndcg > best + 1e-4:
            best = val_ndcg
            _save_fp16_and_report(model, tok, out_dir)

    print(f"\n[RESULT:{slug}] Best VAL NDCG@{params_runtime['VAL_EVAL_K']}: {best:.4f} (baseline {baseline:.4f})")

    # Cleanup GPU RAM before next corpus
    del model, tok, optimizer, scheduler, scaler, train_ds, train_loader, pretok_val
    torch.cuda.empty_cache(); gc.collect()

    return {"baseline": float(baseline), "best": float(best)}


# ---------------------- Run training for selected corpora ----------------------
slugs_to_run = _resolve_run_corpora(RUN_CORPORA)
if not slugs_to_run:
    print("[WARN] Nothing to run. Set RUN_CORPORA = 'all' or a list like ['kz', 'knesset', 'wiki'].")
else:
    print(f"[RUN] corpora: {slugs_to_run}")

summary = {}
for slug in slugs_to_run:
    res = train_one_corpus(slug)
    summary[slug] = res

print("\n================ SUMMARY (per-corpus) ================")
for slug, v in summary.items():
    print(f"{slug:10s}  baseline={v['baseline']:.4f}  best={v['best']:.4f}")
print("======================================================")


| Corpus | Baseline | Best |
|--------|----------|------|
| Wiki | 0.7307 | 0.7412 |
| KZ | 0.5001 | 0.5853 |
| Knesset | 0.4516 | 0.5238 |

## Step 7: Final Stage - Train on Validation Only

Fine-tunes the best checkpoints from Step 6 on **validation data only** for 1 additional epoch.

**Purpose**: Maximize performance on held-out queries before final submission.

Uses same KD + supervised loss setup with conservative hyperparameters (lower LR, higher LOSS_LABELED_ONLY).

In [2]:
# === Cell: Final Stage — train best CEs on VALIDATION ONLY for 1 epoch ===
import os, json, math, time, random, gc
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup

# ---------------------- Base Config ----------------------
OUTPUT_DIR      = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og")

# These are only used for KD + loss params etc.
LR              = float(os.getenv("FINAL_LR", "8e-6"))
BATCH_GROUPS    = int(os.getenv("FINAL_BATCH_GROUPS", "1"))
GRAD_ACCUM      = int(os.getenv("FINAL_GRAD_ACCUM", "2"))
MAX_LEN         = int(os.getenv("FINAL_MAX_LEN", "384"))
WEIGHT_DECAY    = float(os.getenv("FINAL_WEIGHT_DECAY", "0.02"))
CLIP_NORM       = float(os.getenv("FINAL_CLIP_NORM", "1.0"))
TAU             = float(os.getenv("FINAL_TAU", "1.0"))
VAL_EVAL_K      = int(os.getenv("VAL_EVAL_K", "20"))
LOSS_LABELED_ONLY  = int(os.getenv("LOSS_LABELED_ONLY", "1"))  # 1: use only labeled (>0) in sup loss

# KD settings (teacher = Voyage / RRF scores)
ALPHA_SUP       = float(os.getenv("ALPHA_SUP", "0.7"))   # supervised ListNet weight
BETA_KD         = float(os.getenv("BETA_KD",   "0.1"))   # KD ListNet weight
T_KD            = float(os.getenv("T_KD",      "1.8"))   # teacher temperature
KD_MISSING_SENTINEL = float(os.getenv("KD_MISSING_SENTINEL", "nan"))

# Eval/pretokenization
EVAL_BATCH_PAIRS   = int(os.getenv("EVAL_BATCH_PAIRS", "128"))
PAD_TO_MULTIPLE_OF = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))

SEED = int(os.getenv("SEED", "42"))

# Stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, "groups_val_k70.jsonl"))

# Teacher files (produced by RRF+Voy generator)
TEACHER_DIR   = os.getenv("TEACHER_DIR", os.path.join("/content/", "teacher_scores_new"))
TEACHER_FILEPAT = os.getenv("TEACHER_FILEPAT", "{corpus}_teacher_rrf_voy25_k100.jsonl")

# Which corpora to run in this final stage
# "all" or any subset list, e.g., "kz", "knesset", "wiki" or "kz,knesset"
RUN_CORPORA = os.getenv("RUN_CORPORA", "kz")

# ---------------------- Corpus keys & aliases ----------------------
CORPUS_KEYS = {
    "mafat_retrieval_wikipedia_corpus": "wiki",
    "mafat_retrieval_kz_corpus":        "kz",
    "mafat_retrieval_knesset_corpus":   "knesset",
}
SLUG_TO_CORPUSKEY = {v: k for k, v in CORPUS_KEYS.items()}
ALIASES = {"kenesset": "knesset"}  # accept a misspelling

SLUG_TO_TEACHER_CORPUSNAME = {
    "wiki": "wikipedia",
    "kz": "kz",
    "knesset": "knesset",
}

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"[FINAL-CONFIG] device={device}")
print(f"[FINAL-PATHS] OUT={OUTPUT_DIR}  STAGE1={STAGE1_DIR}  TEACHER_DIR={TEACHER_DIR}")
print(f"[FINAL-PATHS] GROUPS_VAL_JSONL={GROUPS_VAL_JSONL}")

# ---------------------- Per-corpus ratings (r_table) ----------------------
R_TABLES_DEFAULT = {
    "wiki":    [0.0, 1.12116936, 5.2045328, 15.2433052, 37.70000983],
    "kz":      [0.0, 1.28305769, 6.25935032, 21.0831492, 42.58293821],
    "knesset": [0.0, 1.12116936, 4.75205225, 16.28861888, 40.28529796],
}

def _load_r_tables(json_path: str) -> Dict[str, List[float]]:
    rt = {k: v[:] for k, v in R_TABLES_DEFAULT.items()}
    try:
        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            for slug in ("wiki", "kz", "knesset"):
                if slug in data and "r_table" in data[slug]:
                    arr = [float(x) for x in data[slug]["r_table"]]
                    if len(arr) == 5 and abs(arr[0]) < 1e-12:
                        rt[slug] = arr
            print(f"[R_TABLES] loaded from {json_path}")
        else:
            print(f"[R_TABLES] file not found, using defaults")
    except Exception as e:
        print(f"[R_TABLES] failed to load ({e}); using defaults")
    return rt

IDEAL_R_JSON = os.path.join(OUTPUT_DIR, "ideal_r_no_training.json")
R_TABLES = _load_r_tables(IDEAL_R_JSON)

# ---------------------- IO ----------------------
def _read_jsonl(path: str):
    items=[]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

groups_val = _read_jsonl(GROUPS_VAL_JSONL)
print(f"[FINAL-DATA] val groups={len(groups_val)}")

# ---------------------- Loss & KD ----------------------
def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0:
            continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

def listnet_loss_labeled_only(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        pos = g > 0
        if pos.sum() == 0:
            continue
        p_t = g[pos] / (g[pos].sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e][pos] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

def listnet_kd_loss(scores: torch.Tensor,
                    t_scores: torch.Tensor,
                    spans,
                    T_kd: float = 1.8,
                    tau_student: float = 1.0):
    scores = scores.float()
    t_scores = t_scores.float()
    loss_terms = []
    for s, e in spans:
        ts = t_scores[s:e]
        mask = torch.isfinite(ts)
        if not torch.any(mask):
            continue
        ts = ts[mask]
        ts = ts - ts.max()
        p_t = torch.softmax(ts / T_kd, dim=0)
        log_q = torch.log_softmax(scores[s:e][mask] / tau_student, dim=0)
        loss_terms.append(-(p_t * log_q).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ---------------------- Teacher loader & attach (KD) ----------------------
def _load_teacher_map_for_slug(slug: str) -> Dict[str, Dict[str, Dict[str, float]]]:
    corpusname = SLUG_TO_TEACHER_CORPUSNAME[slug]
    tpath = os.path.join(TEACHER_DIR, TEACHER_FILEPAT.format(corpus=corpusname))
    if not os.path.exists(tpath):
        print(f"[KD][WARN] teacher file not found for {slug}: {tpath}")
        return {}
    mp = {}
    n = 0
    for rec in _read_jsonl(tpath):
        qid   = rec.get("query_uuid")
        pids  = [str(p) for p in (rec.get("pids") or [])]
        scores = rec.get("teacher_scores") or []
        if not qid or not pids:
            continue
        score_by_doc = {}
        rank_by_doc  = {}
        for idx, pid in enumerate(pids):
            rank_by_doc[pid] = idx
            if idx < len(scores) and scores[idx] is not None:
                score_by_doc[pid] = float(scores[idx])
        mp[qid] = {"score_by_doc": score_by_doc, "rank_by_doc": rank_by_doc}
        n += 1
    print(f"[KD] loaded teacher for {slug}: {n} queries from {tpath}")
    return mp

def _attach_teacher_scores(groups: List[dict], slug: str) -> None:
    tmap = _load_teacher_map_for_slug(slug)
    touched, covered = 0, 0
    for g in groups:
        qid = g.get("query_uuid")
        if not qid:
            g["teacher_scores"] = [float("nan")] * len(g["pids"])
            continue
        rec  = tmap.get(qid, {})
        smap = rec.get("score_by_doc", {})
        ts   = []
        hit  = 0
        for pid in g["pids"]:
            s = smap.get(str(pid))
            if s is None:
                if math.isnan(KD_MISSING_SENTINEL):
                    ts.append(float("nan"))
                else:
                    ts.append(float(KD_MISSING_SENTINEL))
            else:
                ts.append(float(s)); hit += 1
        g["teacher_scores"] = ts
        touched += 1; covered += (hit > 0)
    print(f"[KD] attached teacher_scores to {touched} groups; groups with ≥1 teacher score: {covered}")

# ---------------------- Dataset / Collate ----------------------
class ListwiseDataset(torch.utils.data.Dataset):
    def __init__(self, groups, use_teacher: bool = False):
        self.groups = groups
        self.use_teacher = use_teacher
    def __len__(self): return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        t_scores = g.get("teacher_scores") if self.use_teacher else None
        return g["query"], g["texts"], g["labels"], g["pids"], t_scores

def collate_listwise(batch, tokenizer, max_len=512, r_table=None):
    Q, P, gains, spans, PIDS, TSC = [], [], [], [], [], []
    cur = 0
    use_custom = isinstance(r_table, (list, tuple)) and len(r_table) == 5
    for q, texts, labs, pids, t_scores in batch:
        n = len(texts)
        Q += [q] * n
        P += texts
        if use_custom:
            gains.extend([float(r_table[int(l)]) for l in labs])
        else:
            gains.extend([float((2**int(l)) - 1) for l in labs])
        PIDS += pids
        spans.append((cur, cur + n))
        cur += n
        if t_scores is None:
            TSC += [float("nan")] * n
        else:
            TSC += [float(x) for x in t_scores]
    enc = tokenizer(
        Q, P,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    gains_t = torch.tensor(gains, dtype=torch.float32)
    tsc_t   = torch.tensor(TSC,   dtype=torch.float32)
    return enc, gains_t, spans, PIDS, tsc_t

# ---------------------- Helpers ----------------------
def _filter_by_corpus(groups, corpus_key: str):
    return [g for g in groups if (g.get("case_name") == corpus_key)]

def _resolve_run_corpora(run_spec):
    if isinstance(run_spec, str):
        run_spec = run_spec.strip().lower()
        if run_spec == "all":
            return list(SLUG_TO_CORPUSKEY.keys())
        if "," in run_spec:
            slugs = []
            for x in run_spec.split(","):
                x = x.strip()
                if not x:
                    continue
                y = ALIASES.get(x, x)
                if y in SLUG_TO_CORPUSKEY:
                    slugs.append(y)
            return list(dict.fromkeys(slugs))
        run_spec = ALIASES.get(run_spec, run_spec)
        return [run_spec] if run_spec in SLUG_TO_CORPUSKEY else []
    elif isinstance(run_spec, (list, tuple, set)):
        slugs = []
        for x in run_spec:
            y = ALIASES.get(str(x).lower(), str(x).lower())
            if y in SLUG_TO_CORPUSKEY: slugs.append(y)
        return list(dict.fromkeys(slugs))
    return []

def _dir_size_bytes(path: str) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try: total += os.path.getsize(fp)
            except OSError: pass
    return total

def _human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    i = 0; v = float(n)
    while v >= 1024.0 and i < len(units)-1: v /= 1024.0; i += 1
    return f"{v:.2f} {units[i]}"

def _save_fp16_and_report(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    orig_dtype = next(model.parameters()).dtype
    try:
        model.to(dtype=torch.float16)
        model.save_pretrained(out_dir, safe_serialization=True)
        tokenizer.save_pretrained(out_dir)
    finally:
        model.to(dtype=orig_dtype)
    sz = _dir_size_bytes(out_dir)
    print(f"[FINAL-SAVE] Checkpoint saved (fp16) → {out_dir}  |  folder size = {_human_bytes(sz)}")

# ---------------------- Final train on VAL for one epoch ----------------------
def final_train_on_val(slug: str) -> Dict[str, float]:
    corpus_key = SLUG_TO_CORPUSKEY[slug]
    base_dir   = os.path.join(OUTPUT_DIR, f"ce_{slug}")          # best models from Stage 2
    out_dir    = os.path.join(OUTPUT_DIR, f"ce_{slug}_final")    # final models

    if not os.path.isdir(base_dir):
        print(f"[FINAL:{slug}] base model dir not found: {base_dir}  — skipping.")
        return {"final_train_groups": 0}

    print(f"\n[FINAL:{slug}] base_dir={base_dir}")
    print(f"[FINAL:{slug}] out_dir={out_dir}")

    # Pick r_table for this corpus
    r_table_for_slug = R_TABLES.get(slug)
    if r_table_for_slug is None:
        print(f"[R_TABLES:{slug}] missing; falling back to defaults")
        r_table_for_slug = R_TABLES_DEFAULT.get(slug, [0.0, 1.0, 3.0, 7.0, 15.0])
    print(f"[R_TABLES:{slug}] {r_table_for_slug}")

    # Use VAL groups for this corpus as training data now
    va_all = _filter_by_corpus(groups_val, corpus_key)
    va = [g for g in va_all if any(l > 0 for l in g["labels"])]
    print(f"[FINAL:{slug}] val groups used as train={len(va)} (raw val={len(va_all)})")

    if len(va) == 0:
        print(f"[FINAL:{slug}] No val groups with positives; skipping.")
        return {"final_train_groups": 0}

    # Attach teacher scores (KD) to val groups
    _attach_teacher_scores(va, slug)

    # Load tokenizer & model from best checkpoint
    tok = AutoTokenizer.from_pretrained(base_dir)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            base_dir, trust_remote_code=True, attn_implementation="sdpa"
        ).to(device)
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(
            base_dir, trust_remote_code=True
        ).to(device)
    model = model.float()

    # DataLoader over VAL-as-train
    train_ds = ListwiseDataset(va, use_teacher=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_GROUPS,
        shuffle=True,
        collate_fn=lambda b: collate_listwise(b, tok, MAX_LEN, r_table_for_slug),
        pin_memory=torch.cuda.is_available(),
        num_workers=2 if torch.cuda.is_available() else 0,
        persistent_workers=torch.cuda.is_available(),
    )

    # Optim/sched for exactly 1 epoch
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        eps=1e-8,
        betas=(0.9, 0.999),
    )

    grad_accum = max(1, int(GRAD_ACCUM))
    updates_per_epoch = max(1, math.ceil(len(train_loader) / grad_accum))
    num_update_steps  = updates_per_epoch   # single epoch

    warmup_ratio = 0.10
    warmup = max(1, int(warmup_ratio * num_update_steps))
    lr_end_frac = 0.10
    lr_end = max(LR * lr_end_frac, 1e-8)

    scheduler = get_polynomial_decay_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup,
        num_training_steps=num_update_steps,
        lr_end=lr_end,
        power=1.0,
    )

    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    compute_sup_loss = (listnet_loss_labeled_only if LOSS_LABELED_ONLY else listnet_loss)

    print(f"[FINAL:{slug}] Starting 1 epoch of training on VAL groups …")
    model.train()
    running, micro = 0.0, 0
    t0 = time.perf_counter()

    for step, (enc_cpu, gains_cpu, spans, _pids, t_scores_cpu) in enumerate(train_loader, 1):
        enc      = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        gains    = gains_cpu.to(device, non_blocking=True)
        t_scores = t_scores_cpu.to(device, non_blocking=True)

        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            class _NoOp:
                def __enter__(self): pass
                def __exit__(self, *args): return False
            ctx = _NoOp()

        with ctx:
            logits = model(**enc).logits
            if logits.dim() == 2 and logits.shape[1] == 1:
                s = logits.squeeze(-1).float()
            elif logits.dim() == 2 and logits.shape[1] == 2:
                s = logits[:,1].float()
            else:
                s = logits.view(-1).float()

            L_sup = compute_sup_loss(s, gains, spans, tau=TAU)
            L_kd  = listnet_kd_loss(s, t_scores, spans, T_kd=T_KD, tau_student=TAU)

            loss = ALPHA_SUP * L_sup + BETA_KD * L_kd
            loss = loss / grad_accum

        micro += 1

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        do_step = (step % grad_accum == 0) or (step == len(train_loader))

        if do_step:
            if use_amp:
                if CLIP_NORM > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                if CLIP_NORM > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running += float(loss.item())
        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.perf_counter() - t0
            print(
                f"[FINAL-TRAIN:{slug}] step {step}/{len(train_loader)} "
                f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                f"steps/s={step/max(1e-6,elapsed):.2f}"
            )

    print(f"[FINAL:{slug}] 1-epoch training finished in {time.perf_counter()-t0:.1f}s")
    _save_fp16_and_report(model, tok, out_dir)

    # Cleanup
    del model, tok, optimizer, scheduler, scaler, train_ds, train_loader
    torch.cuda.empty_cache(); gc.collect()

    return {"final_train_groups": len(va)}

# ---------------------- Run final training ----------------------
slugs_to_run = _resolve_run_corpora(RUN_CORPORA)
if not slugs_to_run:
    print("[FINAL] Nothing to run. Set RUN_CORPORA = 'all' or like 'kz,knesset,wiki'.")
else:
    print(f"[FINAL] corpora: {slugs_to_run}")

final_summary = {}
for slug in slugs_to_run:
    res = final_train_on_val(slug)
    final_summary[slug] = res

print("\n================ FINAL TRAIN SUMMARY (per-corpus) ================")
for slug, v in final_summary.items():
    print(f"{slug:10s}  train_on_val_groups={v.get('final_train_groups', 0)}")
print("==================================================================")


[FINAL-CONFIG] device=cuda
[FINAL-PATHS] OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og  STAGE1=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/stage1  TEACHER_DIR=/content/teacher_scores_new
[FINAL-PATHS] GROUPS_VAL_JSONL=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/stage1/groups_val_k70.jsonl
[R_TABLES] file not found, using defaults
[FINAL-DATA] val groups=306
[FINAL] corpora: ['kz']

[FINAL:kz] base_dir=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/ce_kz
[FINAL:kz] out_dir=/content/ce_ft/bge-reranker-v2-m3_t50_fp16_k70_og/ce_kz_final
[R_TABLES:kz] [0.0, 1.28305769, 6.25935032, 21.0831492, 42.58293821]
[FINAL:kz] val groups used as train=119 (raw val=122)
[KD] loaded teacher for kz: 804 queries from /content/teacher_scores_new/kz_teacher_rrf_voy25_k100.jsonl
[KD] attached teacher_scores to 119 groups; groups with ≥1 teacher score: 119
[FINAL:kz] Starting 1 epoch of training on VAL groups …
[FINAL-TRAIN:kz] step 50/119 loss(avg)=0.9454  lr=5.47e-06 steps/s=2.14
[FIN